In [372]:
import warnings
warnings.filterwarnings('ignore')

import os

import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

input_table = 'TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU_NEW'
month_run = '2026-09-30'
os.listdir('/data/aman_singh/acuuracy_check')

['MARICO LIMITED_swiggy_jul.xlsx',
 'Stat Demand Forecast GT_as_on_6th_July_2026.xlsb',
 'soh_data_ecom.csv',
 'anaplan_snapshots_july.xlsx',
 'anaplan_snapshots_aug.xlsx',
 'Stat_forecast_Qcom_as_on_7th_july_2026.xlsx',
 'qcom_chain_forecast_oct.csv',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'Alternate_export_file (2).xlsx',
 'QCOM Chain PSKU OTP Output',
 'all_combination_qcom_aug_pred.csv',
 'key_check.csv',
 'chain_depot_psku_oct.csv',
 'Marico Ltd._forecast_May 2026_to_Aug 2026_blinkit.csv',
 'Stat Demand Forecast MT_as_on_7th_July_2026.xlsb',
 'Ecom_Stat_forecast_chain_psku_as_on_20th_aug_2026.xlsx',
 'ecom_primary_trend.csv',
 'ALL Channels Accuracy_fva.ipynb',
 'all_combination_ecom_aug_pred.csv',
 'offtakes_monthly_aggregated.csv',
 'Stat Demand Forecast MT_as_on_9th_June_2026.xlsb',
 'Marico_Limited_projection_oct_zepto.csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'all_combination_MT_live_aug.csv',
 'seasonality.xlsx',
 'chain_forecast live.ipynb'

In [373]:
base_dir = '/data/aman_singh/acuuracy_check'


In [374]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [375]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_jul.xlsx',
 '/data/aman_singh/acuuracy_check/Stat Demand Forecast GT_as_on_6th_July_2026.xlsb',
 '/data/aman_singh/acuuracy_check/soh_data_ecom.csv',
 '/data/aman_singh/acuuracy_check/anaplan_snapshots_july.xlsx',
 '/data/aman_singh/acuuracy_check/anaplan_snapshots_aug.xlsx',
 '/data/aman_singh/acuuracy_check/Stat_forecast_Qcom_as_on_7th_july_2026.xlsx',
 '/data/aman_singh/acuuracy_check/qcom_chain_forecast_oct.csv',
 '/data/aman_singh/acuuracy_check/combine_model+missing_forecasts_brand_asm.ipynb',
 '/data/aman_singh/acuuracy_check/Alternate_export_file (2).xlsx',
 '/data/aman_singh/acuuracy_check/all_combination_qcom_aug_pred.csv',
 '/data/aman_singh/acuuracy_check/key_check.csv',
 '/data/aman_singh/acuuracy_check/chain_depot_psku_oct.csv',
 '/data/aman_singh/acuuracy_check/Marico Ltd._forecast_May 2026_to_Aug 2026_blinkit.csv',
 '/data/aman_singh/acuuracy_check/Stat Demand Forecast MT_as_on_7th_July_2026.xlsb',
 '/data/aman_sin

In [376]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [377]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [378]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-01' and run_month = '{month_run}'
        
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,VOL_IN_RUM,INDEXBPM,BRAND_CODE,IMPUTED,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RATIO_LAST_YEAR,QUARTER,RUN_MONTH
0,2023-01-31,Amazon ARIPL,718288,9.640,13.27071,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.994832,1,2026-09-30
1,2023-02-28,Amazon ARIPL,718288,7.915,10.89602,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.861631,1,2026-09-30
2,2023-03-31,Amazon ARIPL,718288,9.505,13.08486,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,1.177605,1,2026-09-30
3,2023-04-30,Amazon ARIPL,718288,9.290,12.78889,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.834181,2,2026-09-30
4,2023-05-31,Amazon ARIPL,718288,8.050,11.08187,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,0.957359,2,2026-09-30


In [379]:
offtake_df.columns = offtake_df.columns.str.lower()

In [380]:
offtake_df.columns

Index(['month_date', 'platform_name', 'parent_material_code', 'vol_in_rum',
       'indexbpm', 'brand_code', 'imputed', 'big_billion_days',
       'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2',
       'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'ratio_last_year', 'quarter',
       'run_month'],
      dtype='object')

In [381]:
offtake_df = offtake_df.groupby(['month_date', 'parent_material_code'])[['vol_in_rum',
       'indexbpm']].sum().reset_index()
offtake_df['key'] = offtake_df['parent_material_code']
offtake_df

,month_date,parent_material_code,vol_in_rum,indexbpm,key
0,2023-01-31,709540,924.950,15.11976,709540
1,2023-01-31,709541,206.000,3.36739,709541
2,2023-01-31,715096,6.600,0.08053,715096
3,2023-01-31,715099,2.775,0.03386,715099
4,2023-01-31,715100,4.000,0.04880,715100
...,...,...,...,...,...
43305,2027-06-30,811268,0.000,0.00000,811268
43306,2027-06-30,811269,0.000,0.00000,811269
43307,2027-06-30,811279,0.000,0.00000,811279
43308,2027-06-30,811287,0.000,0.00000,811287


In [382]:
offtake_df.duplicated(
    subset=['parent_material_code', 'month_date']).sum()

0

In [383]:
# offtake_df['brand_code'] = np.where(
#     ((offtake_df['parent_material_code'] == 715096) &
#     (offtake_df['brand_code'] == 'CO_SO_PCP')),
#     'CO_SO_FS',
#     offtake_df['brand_code']
# )

In [384]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [385]:
offtake_df['key'] = offtake_df[['parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [386]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [387]:
(offtake_df['key'] == offtake_df[['parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [388]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [389]:
base_dir

'/data/aman_singh/acuuracy_check'

In [390]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [391]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/trend_file_train_till_31_Mar_2026.csv
/data/aman_singh/acuuracy_check/trend_file_train_till_30_Jun_2026.csv
/data/aman_singh/acuuracy_check/trend_file_train_till_30_Apr_2026.csv
/data/aman_singh/acuuracy_check/trend_file_train_till_28_Feb_2026.csv
/data/aman_singh/acuuracy_check/trend_file_train_till_31_May_2026.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_28_Feb_2026.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_30_Jun_2026.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_Mar_2026.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_May_2026.csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_30_Apr_2026.csv


In [392]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [393]:
# forecast_train_till_file_df

In [394]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,vol_in_rum,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path
0,709538,2023-10-31,0.000000,0.060,0.040,0.089118,0.065207,0.000000,0.000010,0.000007,0.000015,0.000011,709538,0.10,LIVON SLP,1,0,0,0,0,1,0,0,0,0,0.000000,4,1634.657242,0.000016,0.10,0.000016,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
1,709538,2023-11-30,0.081494,0.060,0.040,0.012019,0.044420,0.000013,0.000010,0.000007,0.000002,0.000007,709538,0.02,LIVON SLP,0,1,0,0,0,0,1,0,0,0,0.000000,4,1634.657242,0.000003,0.02,0.000003,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
2,709538,2023-12-31,0.052875,0.060,0.040,0.032824,0.051573,0.000009,0.000010,0.000007,0.000005,0.000008,709538,0.06,LIVON SLP,0,0,1,0,0,0,0,1,0,0,0.600000,4,1634.657242,0.000010,0.06,0.000010,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
3,709538,2024-01-31,0.087334,0.060,0.040,0.024579,0.042147,0.000014,0.000010,0.000007,0.000004,0.000007,709538,0.04,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.666667,1,1634.657242,0.000007,0.04,0.000007,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
4,709538,2024-02-29,-0.017065,0.040,0.040,0.012909,0.013573,-0.000003,0.000007,0.000007,0.000002,0.000002,709538,0.00,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.000000,1,1634.657242,0.000000,0.00,0.000000,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35996,811169,2026-09-30,42.238115,56.376,67.828,111.479256,73.387989,0.007234,0.009655,0.011616,0.019092,0.012568,811169,0.00,SW_SGPRF,0,0,0,1,0,0,0,0,1,0,1.228669,3,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
35997,811169,2026-10-31,40.871633,56.376,67.828,102.365332,53.149755,0.007000,0.009655,0.011616,0.017531,0.009102,811169,0.00,SW_SGPRF,1,0,0,0,0,1,0,0,0,0,0.274970,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
35998,811169,2026-11-30,33.768187,56.376,67.828,55.582370,48.242028,0.005783,0.009655,0.011616,0.009519,0.008262,811169,0.00,SW_SGPRF,0,1,0,0,0,0,1,0,0,0,0.231647,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
35999,811169,2026-12-31,30.944670,56.376,67.828,71.896561,50.040562,0.005300,0.009655,0.011616,0.012313,0.008570,811169,0.00,SW_SGPRF,0,0,1,0,0,0,0,1,0,0,0.533962,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...


In [395]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'vol_in_rum', 'brand_code', 'big_billion_days',
       'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2',
       'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'ratio_last_year', 'quarter',
       'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path'],
      dtype='object')

In [396]:
# trend_file_df['platform_name'].unique()

In [397]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [398]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [399]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-04-30 00:00:00'): {Timestamp('2026-04-30 00:00:00'): 'M',
  Timestamp('2026-05-31 00:00:00'): 'M+1',
  Timestamp('2026-06-30 00:00:00'): 'M+2',
  Timestamp('2026-07-31 00:00:00'): 'M+3',
  Timestamp('2026-08-31 00:00:00'): 'M+4',
  Timestamp('2026-09-30 00:00:00'): 'M+5',
  Timestamp('2026-10-31 00:00:00'): 'M+6',
  Timestamp('2026-11-30 00:00:00'): 'M+7',
  Timestamp('2026-12-31 00:00:00'): 'M+8'},
 Timestamp('2026-07-31 00:00:00'): {Timestamp('2026-07-31 00:00:00'): 'M',
  Timestamp('2026-08-31 00:00:00'): 'M+1',
  Timestamp('2026-09-30 00:00:00'): 'M+2',
  Timestamp('2026-10-31 00:00:00'): 'M+3',
  Timestamp('2026-11-30 00:00:00'): 'M+4',
  Timestamp('2026-12-31 00:00:00'): 'M+5',
  Timestamp('2027-01-31 00:00:00'): 'M+6',
  Timestamp('2027-02-28 00:00:00'): 'M+7',
  Timestamp('2027-03-31 00:00:00'): 'M+8'},
 Timestamp('2026-05-31 00:00:00'): {Timestamp('2026-05-31 00:00:00'): 'M',
  Timestamp('2026-06-30 00:00:00'): 'M+1',
  Timestamp('2026-07-31 00:00:00'): 'M+2',

In [400]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [401]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,vol_in_rum,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
0,709538,2023-10-31,0.000000,0.060,0.040,0.089118,0.065207,0.000000,0.000010,0.000007,0.000015,0.000011,709538,0.10,LIVON SLP,1,0,0,0,0,1,0,0,0,0,0.000000,4,1634.657242,0.000016,0.10,0.000016,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None
1,709538,2023-11-30,0.081494,0.060,0.040,0.012019,0.044420,0.000013,0.000010,0.000007,0.000002,0.000007,709538,0.02,LIVON SLP,0,1,0,0,0,0,1,0,0,0,0.000000,4,1634.657242,0.000003,0.02,0.000003,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None
2,709538,2023-12-31,0.052875,0.060,0.040,0.032824,0.051573,0.000009,0.000010,0.000007,0.000005,0.000008,709538,0.06,LIVON SLP,0,0,1,0,0,0,0,1,0,0,0.600000,4,1634.657242,0.000010,0.06,0.000010,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None
3,709538,2024-01-31,0.087334,0.060,0.040,0.024579,0.042147,0.000014,0.000010,0.000007,0.000004,0.000007,709538,0.04,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.666667,1,1634.657242,0.000007,0.04,0.000007,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None
4,709538,2024-02-29,-0.017065,0.040,0.040,0.012909,0.013573,-0.000003,0.000007,0.000007,0.000002,0.000002,709538,0.00,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.000000,1,1634.657242,0.000000,0.00,0.000000,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35996,811169,2026-09-30,42.238115,56.376,67.828,111.479256,73.387989,0.007234,0.009655,0.011616,0.019092,0.012568,811169,0.00,SW_SGPRF,0,0,0,1,0,0,0,0,1,0,1.228669,3,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3
35997,811169,2026-10-31,40.871633,56.376,67.828,102.365332,53.149755,0.007000,0.009655,0.011616,0.017531,0.009102,811169,0.00,SW_SGPRF,1,0,0,0,0,1,0,0,0,0,0.274970,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4
35998,811169,2026-11-30,33.768187,56.376,67.828,55.582370,48.242028,0.005783,0.009655,0.011616,0.009519,0.008262,811169,0.00,SW_SGPRF,0,1,0,0,0,0,1,0,0,0,0.231647,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+5
35999,811169,2026-12-31,30.944670,56.376,67.828,71.896561,50.040562,0.005300,0.009655,0.011616,0.012313,0.008570,811169,0.00,SW_SGPRF,0,0,1,0,0,0,0,1,0,0,0.533962,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+6


In [402]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,vol_in_rum,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
30,709538,2026-04-30,-0.000080,0.000,0.000,0.000000,0.000000,-1.304874e-08,0.000000,0.000000,0.000000,0.000000,709538,0.0,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.000000,2,1634.657242,0.0,0.0,0.0,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,M
31,709538,2026-05-31,-0.002029,0.000,0.000,0.000000,0.000000,-3.317171e-07,0.000000,0.000000,0.000000,0.000000,709538,0.0,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.000000,2,1634.657242,0.0,0.0,0.0,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,M+1
32,709538,2026-06-30,-0.000296,0.000,0.000,0.000000,0.000000,-4.843880e-08,0.000000,0.000000,0.000000,0.000000,709538,0.0,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.000000,2,1634.657242,0.0,0.0,0.0,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,M+2
33,709538,2026-07-31,-0.001801,0.000,0.000,0.000000,0.000000,-2.944024e-07,0.000000,0.000000,0.000000,0.000000,709538,0.0,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.000000,3,1634.657242,0.0,0.0,0.0,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,M+3
34,709538,2026-08-31,-0.002691,0.000,0.000,0.017708,0.000000,-4.398107e-07,0.000000,0.000000,0.000003,0.000000,709538,0.0,LIVON SLP,0,0,0,0,1,0,0,0,0,1,0.000000,3,1634.657242,0.0,0.0,0.0,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35996,811169,2026-09-30,42.238115,56.376,67.828,111.479256,73.387989,7.233722e-03,0.009655,0.011616,0.019092,0.012568,811169,0.0,SW_SGPRF,0,0,0,1,0,0,0,0,1,0,1.228669,3,1712.605337,0.0,0.0,0.0,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3
35997,811169,2026-10-31,40.871633,56.376,67.828,102.365332,53.149755,6.999698e-03,0.009655,0.011616,0.017531,0.009102,811169,0.0,SW_SGPRF,1,0,0,0,0,1,0,0,0,0,0.274970,4,1712.605337,0.0,0.0,0.0,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4
35998,811169,2026-11-30,33.768187,56.376,67.828,55.582370,48.242028,5.783158e-03,0.009655,0.011616,0.009519,0.008262,811169,0.0,SW_SGPRF,0,1,0,0,0,0,1,0,0,0,0.231647,4,1712.605337,0.0,0.0,0.0,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+5
35999,811169,2026-12-31,30.944670,56.376,67.828,71.896561,50.040562,5.299601e-03,0.009655,0.011616,0.012313,0.008570,811169,0.0,SW_SGPRF,0,0,1,0,0,0,0,1,0,0,0.533962,4,1712.605337,0.0,0.0,0.0,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+6


In [403]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-03-31,2026-02-28
0,2026-04-30,2026-03-31
0,2026-05-31,2026-04-30
0,2026-06-30,2026-05-31
0,2026-07-31,2026-06-30


In [404]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-03-31,2026-02-28
0,2026-04-30,2026-03-31
0,2026-05-31,2026-04-30
0,2026-06-30,2026-05-31
0,2026-07-31,2026-06-30


In [405]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7'],
      dtype=object)

In [406]:
portfolio_query = """SELECT DISTINCT
    MATERIAL_GROUP_CODE AS BRAND_CODE,
    SAP_PORTFOLIO_NAME AS PORTFOLIO
FROM MST_MATERIAL
WHERE COMPANY_CODE = 'MIL'
  AND LATEST_RECORD_IND = '1'
  AND MATERIAL_GROUP_CODE IS NOT NULL
  AND SAP_PORTFOLIO_NAME IS NOT NULL
ORDER BY SAP_PORTFOLIO_NAME, MATERIAL_GROUP_CODE;"""
brand_md_df = pd.read_sql(portfolio_query, prod_conn)

brand_md_df.columns = brand_md_df.columns.str.lower()
brand_md_df

,brand_code,portfolio
0,BD_BDOL_M,BEARDO
1,BD_HRWX_M,BEARDO
2,BRD_BDCOL,BEARDO
3,BRD_BDOIL,BEARDO
4,BRD_BDSPR,BEARDO
...,...,...
319,PURSNS_GM,SKIN CARE
320,PURSNS_ML,SKIN CARE
321,TO_CL_OTG,SKIN CARE
322,TO_CL_OTM,SKIN CARE


In [407]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [408]:
trend_file_df['portfolio'].isna().sum()

0

In [409]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [410]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,trend_70_%ile,trend_75_%ile,additive_terms,additive_terms_lower,additive_terms_upper,big_billion_days,big_billion_days_lower,big_billion_days_upper,big_billion_days_lag_1,big_billion_days_lag_1_lower,big_billion_days_lag_1_upper,big_billion_days_lag_2,big_billion_days_lag_2_lower,big_billion_days_lag_2_upper,big_billion_days_lead_1,big_billion_days_lead_1_lower,big_billion_days_lead_1_upper,big_billion_days_lead_2,big_billion_days_lead_2_lower,big_billion_days_lead_2_upper,extra_regressors_additive,extra_regressors_additive_lower,extra_regressors_additive_upper,great_indian_festival,great_indian_festival_lower,great_indian_festival_upper,great_indian_festival_lag_1,great_indian_festival_lag_1_lower,great_indian_festival_lag_1_upper,great_indian_festival_lag_2,great_indian_festival_lag_2_lower,great_indian_festival_lag_2_upper,great_indian_festival_lead_1,great_indian_festival_lead_1_lower,great_indian_festival_lead_1_upper,great_indian_festival_lead_2,great_indian_festival_lead_2_lower,great_indian_festival_lead_2_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat,key,y,month_date,brand_code,qtr_ind_rate,vol_in_rum,vol_in_rum_value,yhat_value,Model_Run,Model_Type,type,train_till,run,step,file_path,run_month
0,2023-10-31,0.027616,0.076433,0.101657,0.027616,0.027616,0.091360,0.094045,0.095413,0.027616,0.027616,0.027616,0.061486,0.061486,0.061486,0.021973,0.021973,0.021973,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.043946,0.043946,0.043946,0.021973,0.021973,0.021973,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.017540,0.017540,0.017540,0.0,0.0,0.0,0.089103,709538,0.10,2023-10-31,LIVON SLP,1634.657242,0.10,0.000016,0.000015,Yes,prophet,training,2026-02-28,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-03-31
1,2023-11-30,0.026567,-0.000031,0.024594,0.026567,0.026567,0.014849,0.017153,0.018336,0.026567,0.026567,0.026567,-0.014515,-0.014515,-0.014515,0.000000,0.000000,0.000000,-0.033548,-0.033548,-0.033548,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.067095,-0.067095,-0.067095,0.000000,0.000000,0.000000,-0.033548,-0.033548,-0.033548,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.052580,0.052580,0.052580,0.0,0.0,0.0,0.012051,709538,0.02,2023-11-30,LIVON SLP,1634.657242,0.02,0.000003,0.000002,Yes,prophet,training,2026-02-28,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-03-31
2,2023-12-31,0.025482,0.020164,0.046695,0.025482,0.025482,0.035813,0.038783,0.040356,0.025482,0.025482,0.025482,0.008009,0.008009,0.008009,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003650,0.003650,0.003650,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.007300,0.007300,0.007300,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003650,0.003650,0.003650,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000710,0.000710,0.000710,0.0,0.0,0.0,0.033491,709538,0.06,2023-12-31,LIVON SLP,1634.657242,0.06,0.000010,0.000005,Yes,prophet,training,2026-02-28,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-03-31
3,2024-01-31,0.024397,0.012006,0.039058,0.024397,0.024397,0.027900,0.030416,0.032291,0.024397,0.024397,0.024397,0.001143,0.001143,0.001143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001143,0.001143,0.001143,0.0,0.0,0.0,0.025539,709538,0.04,2024-01-31,LIVON SLP,1634.657242,0.04,0.000007,0.000004,Yes,prophet,training,2026-02-28,run,acu

In [411]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile','yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [412]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [413]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [414]:
offtake_df.head()

,month_date,parent_material_code,vol_in_rum,indexbpm,key
0,2023-01-31,709540,924.950,15.11976,709540
1,2023-01-31,709541,206.000,3.36739,709541
2,2023-01-31,715096,6.600,0.08053,715096
3,2023-01-31,715099,2.775,0.03386,715099
4,2023-01-31,715100,4.000,0.04880,715100


In [415]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [416]:
offtake_df

,month_date,parent_material_code,vol_in_rum,indexbpm,key
0,2023-01-31,709540,924.950,15.11976,709540
1,2023-01-31,709541,206.000,3.36739,709541
2,2023-01-31,715096,6.600,0.08053,715096
3,2023-01-31,715099,2.775,0.03386,715099
4,2023-01-31,715100,4.000,0.04880,715100
...,...,...,...,...,...
43305,2027-06-30,811268,0.000,0.00000,811268
43306,2027-06-30,811269,0.000,0.00000,811269
43307,2027-06-30,811279,0.000,0.00000,811279
43308,2027-06-30,811287,0.000,0.00000,811287


In [417]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
# offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])


In [418]:
trend_file_df['key'] = trend_file_df['key'].astype(str)

In [419]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [420]:
trend_file_df.select_dtypes('number').isna().sum()

pred_SARIMA                     835
pred_p3m                          0
pred_p6m                          0
pred_prophet                      0
pred_rf                           0
pred_value_SARIMA               835
pred_value_p3m                    0
pred_value_p6m                    0
pred_value_prophet                0
pred_value_rf                     0
parent_material_code              0
big_billion_days                  0
big_billion_days_lag_1            0
big_billion_days_lag_2            0
big_billion_days_lead_1           0
big_billion_days_lead_2           0
great_indian_festival             0
great_indian_festival_lag_1       0
great_indian_festival_lag_2       0
great_indian_festival_lead_1      0
great_indian_festival_lead_2      0
ratio_last_year                   0
quarter                           0
qtr_ind_rate                      0
vol_in_rum_value                  0
vol_in_rum_treated                0
vol_in_rum_value_treated          0
cov                         

In [421]:
trend_file_df.select_dtypes('number').min().round()

pred_SARIMA                     -24689.0
pred_p3m                             0.0
pred_p6m                             0.0
pred_prophet                         0.0
pred_rf                              0.0
pred_value_SARIMA                   -1.0
pred_value_p3m                       0.0
pred_value_p6m                       0.0
pred_value_prophet                   0.0
pred_value_rf                        0.0
parent_material_code            709538.0
big_billion_days                     0.0
big_billion_days_lag_1               0.0
big_billion_days_lag_2               0.0
big_billion_days_lead_1              0.0
big_billion_days_lead_2              0.0
great_indian_festival                0.0
great_indian_festival_lag_1          0.0
great_indian_festival_lag_2          0.0
great_indian_festival_lead_1         0.0
great_indian_festival_lead_2         0.0
ratio_last_year                      0.0
quarter                              1.0
qtr_ind_rate                         0.0
vol_in_rum_value

In [422]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [423]:
for col in [ 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [424]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,709538,2023-10-31,0.000000,0.060,0.040,0.089118,0.065207,0.000000,0.000010,0.000007,0.000015,0.000011,709538,LIVON SLP,1,0,0,0,0,1,0,0,0,0,0.000000,4,1634.657242,0.000016,0.10,0.000016,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None,PREM. HAIR NOUR.,0.094326,0.091785,0.10
1,709538,2023-11-30,0.081494,0.060,0.040,0.012019,0.044420,0.000013,0.000010,0.000007,0.000002,0.000007,709538,LIVON SLP,0,1,0,0,0,0,1,0,0,0,0.000000,4,1634.657242,0.000003,0.02,0.000003,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None,PREM. HAIR NOUR.,0.017181,0.014169,0.02
2,709538,2023-12-31,0.052875,0.060,0.040,0.032824,0.051573,0.000009,0.000010,0.000007,0.000005,0.000008,709538,LIVON SLP,0,0,1,0,0,0,0,1,0,0,0.600000,4,1634.657242,0.000010,0.06,0.000010,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None,PREM. HAIR NOUR.,0.038677,0.035541,0.06
3,709538,2024-01-31,0.087334,0.060,0.040,0.024579,0.042147,0.000014,0.000010,0.000007,0.000004,0.000007,709538,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.666667,1,1634.657242,0.000007,0.04,0.000007,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None,PREM. HAIR NOUR.,0.029644,0.027305,0.04
4,709538,2024-02-29,-0.017065,0.040,0.040,0.012909,0.013573,-0.000003,0.000007,0.000007,0.000002,0.000002,709538,LIVON SLP,0,0,0,0,0,0,0,0,0,0,0.000000,1,1634.657242,0.000000,0.00,0.000000,2026-03-31,2.754307,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,None,PREM. HAIR NOUR.,0.017783,0.015256,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175084,811169,2026-09-30,42.238115,56.376,67.828,111.479256,73.387989,0.007234,0.009655,0.011616,0.019092,0.012568,811169,SW_SGPRF,0,0,0,1,0,0,0,0,1,0,1.228669,3,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,MALE GROOMING,122.229163,116.387287,0.00
175085,811169,2026-10-31,40.871633,56.376,67.828,102.365332,53.149755,0.007000,0.009655,0.011616,0.017531,0.009102,811169,SW_SGPRF,1,0,0,0,0,1,0,0,0,0,0.274970,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4,MALE GROOMING,113.547543,107.114369,0.00
175086,811169,2026-11-30,33.768187,56.376,67.828,55.582370,48.242028,0.005783,0.009655,0.011616,0.009519,0.008262,811169,SW_SGPRF,0,1,0,0,0,0,1,0,0,0,0.231647,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+5,MALE GROOMING,67.234422,61.255213,0.00
175087,811169,2026-12-31,30.944670,56.376,67.828,71.896561,50.040562,0.005300,0.009655,0.011616,0.012313,0.008570,811169,SW_SGPRF,0,0,1,0,0,0,0,1,0,0,0.533962,4,1712.605337,0.000000,0.00,0.000000,2026-05-31,0.635823,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+6,MALE GROOMING,82.562372,76.468134,0.00


In [425]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [426]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [427]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [428]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [429]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [430]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [431]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [432]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [433]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [434]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [435]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [436]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [437]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [438]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_SARIMA',
 'pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'vol_in_rum_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [439]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

pred_value_SARIMA


In [440]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
109222,718471,2023-01-31,-3.077027e-01,10.230,9.4875,8.218025,10.6465,-1.528755e-05,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.297990,1,496.828458,0.000566,11.385,0.000566,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000462,0.000433
109223,718471,2023-02-28,1.107730e+01,10.230,9.4875,7.732822,10.1110,5.503517e-04,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.136752,1,496.828458,0.000503,10.125,0.000503,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000441,0.000415
109224,718471,2023-03-31,9.817297e+00,10.230,9.4875,6.474450,9.4140,4.877513e-04,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.410374,1,496.828458,0.000456,9.180,0.000456,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000380,0.000353
109225,718471,2023-04-30,8.872297e+00,10.230,9.4875,6.578607,9.4800,4.408010e-04,0.000508,0.000471,0.000327,0.000471,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.460962,2,496.828458,0.000461,9.270,0.000461,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.687533,7.087275,9.270,10.230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.230,0.000508,NaN,NaN,NaN,0.000382,0.000352
109226,718471,2023-05-31,8.962297e+00,9.525,9.4875,6.540819,9.2045,4.452724e-04,0.000473,0.000471,0.000325,0.000457,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.475073,2,496.828458,0.000483,9.720,0.000483,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.726867,7.174922,9.720,9.525,NaN,NaN,NaN,NaN,NaN,NaN,-6.891496,NaN,NaN,NaN,9.525,0.000473,NaN,NaN,NaN,0.000384,0.000356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43872,719192,2026-10-31,-1.258270e-309,0.000,0.0000,0.000000,0.0000,-1.258270e-314,0.000000,0.000000,0.000000,0.000000,719192,VEG_CLEAN,1,0,0,0,0,1,0,0,0,0,0.000000,4,100.000000,0.000000,0.000,0.000000,2026-06-30,4.665446,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,M+3,HEALTH & HYGIENE,0.000000,0.000000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.000,0.000000,0.0,0.0,0.0,0.000000,0.000000
43873,719192,2026-11-30,-5.203838e-315,0.000,0.0000,0.000000,0.0000,-5.203993e-320,0.000000,0.000000,0.000000,0.000000,719192,VEG_CLEAN,0,1,0,0,0,0,1,0,0,0,0.000000,4,100.000000,0.000000,0.000,0.000000,2026-06-30,4.665446,run,acuuracy_check,/data/aman_singh/acuuracy_check/t

In [441]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [442]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [443]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [444]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'brand_code', 'big_billion_days',
       'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2',
       'great_indian_festival', 'great_indian_festival_lag_1',
       'great_indian_festival_lag_2', 'great_indian_festival_lead_1',
       'great_indian_festival_lead_2', 'ratio_last_year', 'quarter',
       'qtr_ind_rate', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth

In [445]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [446]:
trend_file_df.reset_index(drop=True, inplace=True)

In [447]:
trend_file_df.shape

(175089, 66)

In [448]:
trend_file_df['key'].nunique()

838

In [449]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [450]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()

In [451]:
# brand_class_df.columns = ['brand_code', 'class']


In [452]:
# len_before_merge = len(trend_file_df)
# trend_file_df = trend_file_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(trend_file_df)
# del len_before_merge

In [453]:
# trend_file_df['class'].isna().sum()

In [454]:
# trend_file_df['class'].unique()

missing combinations

In [455]:
# model_file = pd.read_csv("/data/aman_singh/acuuracy_check/Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv")
# model_file

In [456]:
model_file = trend_file_df.copy()

In [457]:
model_file['key'].nunique()

838

In [458]:
run_month

Timestamp('2026-06-30 00:00:00')

In [459]:
data_query = f"""
    select * from TRN_DF_ECOM_OFFTAKE_PSKU
    where month_date >= '2023-01-01' 
        
"""
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,INDEXBPM,BIG_BILLION_DAYS,BIG_BILLION_DAYS_LAG_1,BIG_BILLION_DAYS_LAG_2,BIG_BILLION_DAYS_LEAD_1,BIG_BILLION_DAYS_LEAD_2,GREAT_INDIAN_FESTIVAL,GREAT_INDIAN_FESTIVAL_LAG_1,GREAT_INDIAN_FESTIVAL_LAG_2,GREAT_INDIAN_FESTIVAL_LEAD_1,GREAT_INDIAN_FESTIVAL_LEAD_2,RATIO_LAST_YEAR,QUARTER,RUN_MONTH
0,2023-10-31,709538,LIVON SLP,0.10,0.00163,1,0,0,0,0,1,0,0,0,0,0.000000,4,2026-03-31
1,2023-11-30,709538,LIVON SLP,0.02,0.00033,0,1,0,0,0,0,1,0,0,0,0.000000,4,2026-03-31
2,2023-12-31,709538,LIVON SLP,0.06,0.00098,0,0,1,0,0,0,0,1,0,0,0.600000,4,2026-03-31
3,2024-01-31,709538,LIVON SLP,0.04,0.00065,0,0,0,0,0,0,0,0,0,0,0.666667,1,2026-03-31
4,2024-02-29,709538,LIVON SLP,0.00,0.00000,0,0,0,0,0,0,0,0,0,0,0.000000,1,2026-03-31


In [460]:
qcom_df.columns = qcom_df.columns.str.lower()

In [461]:
qcom_df['key'] = qcom_df[[ 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [462]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [463]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key', 'run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,parent_material_code,brand_code,vol_in_rum,indexbpm,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month,key,LY
619,2026-05-31,718289,H&C,0.0,0.01744,0,0,0,0,0,0,0,0,0,0,0.000000,2,2026-03-31,718289,NaN
620,2026-06-30,718289,H&C,0.0,0.06170,0,0,0,0,0,0,0,0,0,0,0.000000,2,2026-03-31,718289,NaN
621,2026-07-31,718289,H&C,0.0,0.00698,0,0,0,0,0,0,0,0,0,0,0.400000,3,2026-03-31,718289,NaN
622,2026-08-31,718289,H&C,0.0,0.00232,0,0,0,0,1,0,0,0,0,1,0.058824,3,2026-03-31,718289,NaN
623,2026-09-30,718289,H&C,0.0,0.00000,0,0,0,1,0,0,0,0,1,0,0.000000,3,2026-03-31,718289,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196585,2026-12-31,811416,SAF-MUSLI,0.0,0.00000,0,0,1,0,0,0,0,1,0,0,0.000000,4,2026-07-31,811416,NaN
196586,2027-01-31,811416,SAF-MUSLI,0.0,0.00000,0,0,0,0,0,0,0,0,0,0,0.000000,1,2026-07-31,811416,NaN
196587,2027-02-28,811416,SAF-MUSLI,0.0,0.00000,0,0,0,0,0,0,0,0,0,0,0.000000,1,2026-07-31,811416,NaN
196588,2027-03-31,811416,SAF-MUSLI,0.0,0.00000,0,0,0,0,0,0,0,0,0,0,0.000000,1,2026-07-31,811416,NaN


In [464]:
missing_df['key'].nunique()

201

In [465]:
missing_df = missing_df[['key','run_month','month_date','parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,parent_material_code,brand_code,vol_in_rum
619,718289,2026-03-31,2026-05-31,718289,H&C,0.0
620,718289,2026-03-31,2026-06-30,718289,H&C,0.0
621,718289,2026-03-31,2026-07-31,718289,H&C,0.0
622,718289,2026-03-31,2026-08-31,718289,H&C,0.0
623,718289,2026-03-31,2026-09-30,718289,H&C,0.0
...,...,...,...,...,...,...
196585,811416,2026-07-31,2026-12-31,811416,SAF-MUSLI,0.0
196586,811416,2026-07-31,2027-01-31,811416,SAF-MUSLI,0.0
196587,811416,2026-07-31,2027-02-28,811416,SAF-MUSLI,0.0
196588,811416,2026-07-31,2027-03-31,811416,SAF-MUSLI,0.0


In [466]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PLX_FCWS,1187555.000
1,2027-03-31,CMX_WELPD,3014.000
2,2027-03-31,4700_BCPC,222.000
3,2027-03-31,CMX_PRTPD,3014.000
4,2027-03-31,PA_CN_HGO,488.152


In [467]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [468]:
missing_df

,key,run_month,month_date,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,718289,2026-03-31,2026-05-31,718289,H&C,0.0,388.076436
1,718289,2026-03-31,2026-06-30,718289,H&C,0.0,388.076436
2,718289,2026-03-31,2026-07-31,718289,H&C,0.0,388.076436
3,718289,2026-03-31,2026-08-31,718289,H&C,0.0,388.076436
4,718289,2026-03-31,2026-09-30,718289,H&C,0.0,388.076436
...,...,...,...,...,...,...,...
13352,811416,2026-07-31,2026-12-31,811416,SAF-MUSLI,0.0,315513.490535
13353,811416,2026-07-31,2027-01-31,811416,SAF-MUSLI,0.0,315513.490535
13354,811416,2026-07-31,2027-02-28,811416,SAF-MUSLI,0.0,315513.490535
13355,811416,2026-07-31,2027-03-31,811416,SAF-MUSLI,0.0,315513.490535


In [469]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 11):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)


len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)

In [470]:
missing_df['M month'].unique()

array(['M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8', 'M+9', None, 'M',
       'M+1'], dtype=object)

In [471]:
# offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [472]:
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=3).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [473]:

missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

# len_before_merge = len(missing_df)
# missing_df = missing_df.merge(
#     brand_class_df, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(missing_df)
# del len_before_merge
# missing_df['class'].isna().sum()
# missing_df['class'].unique()

LY P3M_value


In [474]:
pd.set_option('display.max_columns', None)

In [475]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [476]:
model_file

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,718471,2023-01-31,-3.077027e-01,10.230,9.4875,8.218025,10.6465,-1.528755e-05,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.297990,1,496.828458,0.000566,11.385,0.000566,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000462,0.000433,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,718471,2023-02-28,1.107730e+01,10.230,9.4875,7.732822,10.1110,5.503517e-04,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.136752,1,496.828458,0.000503,10.125,0.000503,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000441,0.000415,NaN,NaN,NaN,NaN,0.000566,NaN,NaN
2,718471,2023-03-31,9.817297e+00,10.230,9.4875,6.474450,9.4140,4.877513e-04,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.410374,1,496.828458,0.000456,9.180,0.000456,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000380,0.000353,NaN,NaN,NaN,NaN,0.000503,0.000566,NaN
3,718471,2023-04-30,8.872297e+00,10.230,9.4875,6.578607,9.4800,4.408010e-04,0.000508,0.000471,0.000327,0.000471,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.460962,2,496.828458,0.000461,9.270,0.000461,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.687533,7.087275,9.270,10.230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.230,0.000508,NaN,NaN,NaN,0.000382,0.000352,NaN,NaN,NaN,NaN,0.000456,0.000503,0.000566
4,718471,2023-05-31,8.962297e+00,9.525,9.4875,6.540819,9.2045,4.452724e-04,0.000473,0.000471,0.000325,0.000457,718471,ADV-AHO-R,0,0,0,0,0,0,0,0,0,0,0.475073,2,496.828458,0.000483,9.720,0.000483,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.726867,7.174922,9.720,9.525,NaN,NaN,NaN,NaN,NaN,NaN,-6.891496,NaN,NaN,NaN,9.525,0.000473,NaN,NaN,NaN,0.000384,0.000356,NaN,NaN,NaN,NaN,0.000461,0.000456,0.000503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175084,719192,2026-10-31,-1.258270e-309,0.000,0.0000,0.000000,0.0000,-1.258270e-314,0.000000,0.000000,0.000000,0.000000,719192,VEG_CLEAN,1,0,0,0,0,1,0,0,0,0,0.000000,4,100.000000,0.000000,0.000,0.000000,2026-06-30,4.665446,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-07-31,M+3,HEALTH & HYGIENE,0.000000,0.000000,0.000,0.000,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.000,0.000000,0.0,0.0,0.0,0.000000,0.000

In [477]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,718471,2023-01-31,-0.307703,10.230,9.4875,8.218025,10.6465,-0.000015,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.297990,1.0,496.828458,0.000566,11.385,0.000566,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000462,0.000433,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,718471,2023-02-28,11.077297,10.230,9.4875,7.732822,10.1110,0.000550,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.136752,1.0,496.828458,0.000503,10.125,0.000503,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000441,0.000415,NaN,NaN,NaN,NaN,0.000566,NaN,NaN,0
2,718471,2023-03-31,9.817297,10.230,9.4875,6.474450,9.4140,0.000488,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.410374,1.0,496.828458,0.000456,9.180,0.000456,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000380,0.000353,NaN,NaN,NaN,NaN,0.000503,0.000566,NaN,0
3,718471,2023-04-30,8.872297,10.230,9.4875,6.578607,9.4800,0.000441,0.000508,0.000471,0.000327,0.000471,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.460962,2.0,496.828458,0.000461,9.270,0.000461,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.687533,7.087275,9.270,10.230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.230,0.000508,NaN,NaN,NaN,0.000382,0.000352,NaN,NaN,NaN,NaN,0.000456,0.000503,0.000566,0
4,718471,2023-05-31,8.962297,9.525,9.4875,6.540819,9.2045,0.000445,0.000473,0.000471,0.000325,0.000457,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.475073,2.0,496.828458,0.000483,9.720,0.000483,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.726867,7.174922,9.720,9.525,NaN,NaN,NaN,NaN,NaN,NaN,-6.891496,NaN,NaN,NaN,9.525,0.000473,NaN,NaN,NaN,0.000384,0.000356,NaN,NaN,NaN,NaN,0.000461,0.000456,0.000503,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13352,731338,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,731338,SW STLDEO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,357.647957,0.000000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-07-31,M+5,MALE GROOMING,NaN,NaN,0.000,0.800,0.6,NaN,NaN,NaN,1.05,0.85,-23.809524,61.538462,62.5,2.0,0.925,0.000029,0.000021,NaN,NaN,NaN,NaN,0.00,NaN,0.000000,NaN,0.000011,0.000048

In [478]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,718471,2023-01-31,-0.307703,10.230,9.4875,8.218025,10.6465,-0.000015,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.297990,1.0,496.828458,0.000566,11.385,0.000566,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,0.000,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.000000,0.000462,0.000433,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0
1,718471,2023-02-28,11.077297,10.230,9.4875,7.732822,10.1110,0.000550,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.136752,1.0,496.828458,0.000503,10.125,0.000503,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,0.000,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.000000,0.000441,0.000415,0.00,0.0,0.000000,0.0,0.000566,0.000000,0.000000,0
2,718471,2023-03-31,9.817297,10.230,9.4875,6.474450,9.4140,0.000488,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.410374,1.0,496.828458,0.000456,9.180,0.000456,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,0.000,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.000000,0.000380,0.000353,0.00,0.0,0.000000,0.0,0.000503,0.000566,0.000000,0
3,718471,2023-04-30,8.872297,10.230,9.4875,6.578607,9.4800,0.000441,0.000508,0.000471,0.000327,0.000471,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.460962,2.0,496.828458,0.000461,9.270,0.000461,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.687533,7.087275,9.270,10.230,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,10.230,0.000508,0.000000,0.0,0.000000,0.000382,0.000352,0.00,0.0,0.000000,0.0,0.000456,0.000503,0.000566,0
4,718471,2023-05-31,8.962297,9.525,9.4875,6.540819,9.2045,0.000445,0.000473,0.000471,0.000325,0.000457,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.475073,2.0,496.828458,0.000483,9.720,0.000483,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.726867,7.174922,9.720,9.525,0.0,0.0,0.000,0.00,0.00,0.00,-6.891496,0.000000,0.0,0.0,9.525,0.000473,0.000000,0.0,0.000000,0.000384,0.000356,0.00,0.0,0.000000,0.0,0.000461,0.000456,0.000503,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13352,731338,2026-12-31,0.000000,0.000,0.0000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,731338,SW STLDEO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.

In [479]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,718471,2023-01-31,-0.307703,10.230,9.4875,8.218025,10.6465,-0.000015,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.297990,1.0,496.828458,0.000566,11.385,0.000566,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,0.000,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.000000,0.000462,0.000433,0.00,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0
1,718471,2023-02-28,11.077297,10.230,9.4875,7.732822,10.1110,0.000550,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.136752,1.0,496.828458,0.000503,10.125,0.000503,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,0.000,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.000000,0.000441,0.000415,0.00,0.0,0.000000,0.0,0.000566,0.000000,0.000000,0
2,718471,2023-03-31,9.817297,10.230,9.4875,6.474450,9.4140,0.000488,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.410374,1.0,496.828458,0.000456,9.180,0.000456,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,0.000,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,0.000,0.000000,0.000000,0.0,0.000000,0.000380,0.000353,0.00,0.0,0.000000,0.0,0.000503,0.000566,0.000000,0
3,718471,2023-04-30,8.872297,10.230,9.4875,6.578607,9.4800,0.000441,0.000508,0.000471,0.000327,0.000471,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.460962,2.0,496.828458,0.000461,9.270,0.000461,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.687533,7.087275,9.270,10.230,0.0,0.0,0.000,0.00,0.00,0.00,0.000000,0.000000,0.0,0.0,10.230,0.000508,0.000000,0.0,0.000000,0.000382,0.000352,0.00,0.0,0.000000,0.0,0.000456,0.000503,0.000566,0
4,718471,2023-05-31,8.962297,9.525,9.4875,6.540819,9.2045,0.000445,0.000473,0.000471,0.000325,0.000457,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.475073,2.0,496.828458,0.000483,9.720,0.000483,2026-02-28,1.54713,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.726867,7.174922,9.720,9.525,0.0,0.0,0.000,0.00,0.00,0.00,-6.891496,0.000000,0.0,0.0,9.525,0.000473,0.000000,0.0,0.000000,0.000384,0.000356,0.00,0.0,0.000000,0.0,0.000461,0.000456,0.000503,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13352,731338,2026-12-31,0.000000,0.000,0.0000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,731338,SW STLDEO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.

Heuristic new approac

In [480]:
# pip install pymannkendall

identify events

In [481]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])

In [482]:
import numpy as np 
import pandas as pd 
EVENT_MONTHS = [9, 10, 11] # Sep, Oct, Nov
def detect_event_months(df_grp):
    df_grp = df_grp.sort_values("month_date").copy()
    run_month = df_grp["run_month"].max()

    # only historical data
    hist = df_grp[df_grp["month_date"] < run_month].copy()

    # initialize
    df_grp["event_month_flag"] = 0
    df_grp["event_uplift_factor"] = 0.0

    if len(hist) < 12:
        df_grp["event_sensitive_flag"] = 0
        return df_grp

    event_sensitive = 0

    # loop year-wise
    for year in hist["month_date"].dt.year.unique():

        year_df = hist[hist["month_date"].dt.year == year]

        for _, row in year_df.iterrows():

            month = row["month_date"].month

            if month not in EVENT_MONTHS:
                continue

            curr_date = row["month_date"]

            # previous 12 months before this month
            prev_12m = hist[
                (hist["month_date"] < curr_date) &
                (hist["month_date"] >= curr_date - pd.DateOffset(months=12)) &
                (~hist["month_date"].dt.month.isin(EVENT_MONTHS))  # 
            ]

            if len(prev_12m) < 6:
                continue

            prev_12m_avg = prev_12m["vol_in_rum_value"].mean()

            if prev_12m_avg <= 0 or np.isnan(prev_12m_avg):
                continue

            uplift = row["vol_in_rum_value"] / prev_12m_avg

            if uplift > 2:
                mask = df_grp["month_date"] == curr_date
                df_grp.loc[mask, "event_month_flag"] = 1
                df_grp.loc[mask, "event_uplift_factor"] = uplift
                event_sensitive = 1

    df_grp["event_sensitive_flag"] = event_sensitive
    return df_grp


In [483]:
final_df = (
    final_df
    .groupby(
        ["parent_material_code", "run_month"],
        group_keys=False
    )
    .apply(detect_event_months)
)


In [484]:
# final_df[(final_df['event_month_flag'] == 1) & (final_df['platform_name'] == 'Flipkart National') & ((final_df['parent_material_code'] == 718939))]

In [485]:
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["event_month_flag"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()


In [486]:
adj_df = final_df.groupby(
    [ "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_adj_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()

final_df = final_df.merge(
    adj_df,
    on=["parent_material_code", "run_month"],
    how="left"
)

In [487]:
adj_ly_df = final_df.groupby(
    [ "parent_material_code", "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_adj": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_adj": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_adj_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_adj_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

In [488]:
final_df = final_df.merge(
    adj_ly_df,
    on=["parent_material_code", "run_month"],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value
0,718471,2023-01-31,-3.077027e-01,10.23,9.4875,8.218025,10.6465,-1.528755e-05,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.297990,1.0,496.828458,0.000566,11.385,0.000566,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000462,0.000433,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,807039,2026-06-30,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.000986,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+3,SKIN CARE,0.000000,0.000000,2.600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,718471,2023-02-28,1.107730e+01,10.23,9.4875,7.732822,10.1110,5.503517e-04,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.136752,1.0,496.828458,0.000503,10.125,0.000503,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000441,0.000415,0.0,0.0,0.0,0.0,0.000566,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,807039,2026-07-31,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.003791,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+4,SKIN CARE,0.000000,0.000000,10.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,718471,2023-03-31,9.817297e+00,10.23,9.4875,6.474450,9.4140,4.877513e-04,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.410374,1.0,496.828458,0.000456,9.180,0.000456,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000380,0.000353,0.0,0.0,0.0,0.0,0.000503,0.000566,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188441,719192,2026-10-31,-1.258270e-309,0.00,0.0000,0.000000,0.0000,-1.

In [489]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(final_df)

final_df = final_df.merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(final_df)


Credentials retrieved successfully for prod db.


In [490]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [491]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,709538,2026-03-31,-1,0,0
1,709538,2026-04-30,-1,0,0
2,709538,2026-05-31,-1,0,0
3,709538,2026-06-30,-1,0,0
4,709538,2026-07-31,-1,0,0
...,...,...,...,...,...
4985,811416,2026-03-31,0,0,0
4986,811416,2026-04-30,0,0,0
4987,811416,2026-05-31,0,0,0
4988,811416,2026-06-30,0,0,0


## detect seasonality

In [492]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    [ "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,brand_code,run_month,seasonality_flag
0,ADV-AHO-R,2026-03-31,0
1,ADV-AHO-R,2026-04-30,0
2,ADV-AHO-R,2026-05-31,0
3,ADV-AHO-R,2026-06-30,0
4,ADV-AHO-R,2026-07-31,0
...,...,...,...
660,VEG_CLEAN,2026-03-31,0
661,VEG_CLEAN,2026-04-30,0
662,VEG_CLEAN,2026-05-31,0
663,VEG_CLEAN,2026-06-30,0


In [493]:
# seasonality_df.to_csv('seasonal_ecom.csv')

In [494]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,709538,2026-03-31,0.0,0.0,0.0,0.0
1,709538,2026-04-30,0.0,0.0,0.0,0.0
2,709538,2026-05-31,0.0,0.0,0.0,0.0
3,709538,2026-06-30,0.0,0.0,0.0,0.0
4,709538,2026-07-31,0.0,0.0,0.0,0.0


In [495]:
trend_df = trend_df.merge(threshold_df, on = ['parent_material_code', 'run_month'], how = 'left')
trend_df

,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,709538,2026-03-31,-1,0,0,0.0,0.0,0.0,0.0
1,709538,2026-04-30,-1,0,0,0.0,0.0,0.0,0.0
2,709538,2026-05-31,-1,0,0,0.0,0.0,0.0,0.0
3,709538,2026-06-30,-1,0,0,0.0,0.0,0.0,0.0
4,709538,2026-07-31,-1,0,0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
4985,811416,2026-03-31,0,0,0,0.0,0.0,0.0,0.0
4986,811416,2026-04-30,0,0,0,0.0,0.0,0.0,0.0
4987,811416,2026-05-31,0,0,0,0.0,0.0,0.0,0.0
4988,811416,2026-06-30,0,0,0,0.0,0.0,0.0,0.0


In [496]:
# trend_df.to_csv('t_s_t_df_ecom.csv')

In [497]:
final_df = final_df.merge(seasonality_df, on = ["brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag
0,718471,2023-01-31,-3.077027e-01,10.23,9.4875,8.218025,10.6465,-1.528755e-05,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.297990,1.0,496.828458,0.000566,11.385,0.000566,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000462,0.000433,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0
1,807039,2026-06-30,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.000986,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+3,SKIN CARE,0.000000,0.000000,2.600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3790.533503,0
2,718471,2023-02-28,1.107730e+01,10.23,9.4875,7.732822,10.1110,5.503517e-04,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.136752,1.0,496.828458,0.000503,10.125,0.000503,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000441,0.000415,0.0,0.0,0.0,0.0,0.000566,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0
3,807039,2026-07-31,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.003791,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+4,SKIN CARE,0.000000,0.000000,10.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3790.533503,0
4,718471,2023-03-31,9.817297e+00,10.23,9.4875,6.474450,9.4140,4.877513e-04,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.410374,1.0,496.828458,0.000456,9.180,0.000456,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000380,0.000353,0.0,0.0,0.0,0.0,0.000503,0.000566,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [498]:
trend_df.columns

Index(['parent_material_code', 'run_month', 'trend_flag', 'p3m_slope_flag',
       'final_trend', 'lower_threshold', 'upper_threshold', 'mean_value',
       'std_value'],
      dtype='object')

In [499]:
final_df = final_df.merge(trend_df[[ 'parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = [ "parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,718471,2023-01-31,-3.077027e-01,10.23,9.4875,8.218025,10.6465,-1.528755e-05,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.297990,1.0,496.828458,0.000566,11.385,0.000566,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000462,0.000433,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.0
1,807039,2026-06-30,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.000986,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+3,SKIN CARE,0.000000,0.000000,2.600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3790.533503,0,0,0.0,0.0
2,718471,2023-02-28,1.107730e+01,10.23,9.4875,7.732822,10.1110,5.503517e-04,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.136752,1.0,496.828458,0.000503,10.125,0.000503,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000441,0.000415,0.0,0.0,0.0,0.0,0.000566,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.0
3,807039,2026-07-31,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.003791,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+4,SKIN CARE,0.000000,0.000000,10.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3790.533503,0,0,0.0,0.0
4,718471,2023-03-31,9.817297e+00,10.23,9.4875,6.474450,9.4140,4.877513e-04,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.410374,1.0,496.828458,0.000456,9.180,0.000456,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000380,0.000353,0.0,0.0,0.0,0.0,0.000503,0.000566,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [500]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,718471,2023-01-31,-3.077027e-01,10.23,9.4875,8.218025,10.6465,-1.528755e-05,0.000508,0.000471,0.000408,0.000529,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.297990,1.0,496.828458,0.000566,11.385,0.000566,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,9.305672,8.724734,11.385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000462,0.000433,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.0
1,807039,2026-06-30,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.000986,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+3,SKIN CARE,0.000000,0.000000,2.600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3790.533503,0,0,0.0,0.0
2,718471,2023-02-28,1.107730e+01,10.23,9.4875,7.732822,10.1110,5.503517e-04,0.000508,0.000471,0.000384,0.000502,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.136752,1.0,496.828458,0.000503,10.125,0.000503,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,8.866336,8.354395,10.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000441,0.000415,0.0,0.0,0.0,0.0,0.000566,0.000000,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.0
3,807039,2026-07-31,0.000000e+00,0.00,0.0000,0.000000,0.0000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,807039,BIO OILS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3790.533503,0.003791,0.000,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-03-31,M+4,SKIN CARE,0.000000,0.000000,10.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,1,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3790.533503,0,0,0.0,0.0
4,718471,2023-03-31,9.817297e+00,10.23,9.4875,6.474450,9.4140,4.877513e-04,0.000508,0.000471,0.000322,0.000468,718471,ADV-AHO-R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.410374,1.0,496.828458,0.000456,9.180,0.000456,2026-02-28,1.547130,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,HAIR OILS,7.655366,7.112258,9.180,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000380,0.000353,0.0,0.0,0.0,0.0,0.000503,0.000566,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.828458,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [501]:
final_df.drop(columns = ['big_billion_days', 'big_billion_days_lag_1', 'big_billion_days_lag_2',
       'big_billion_days_lead_1', 'big_billion_days_lead_2', 'great_indian_festival',
       'great_indian_festival_lag_1', 'great_indian_festival_lag_2',
       'great_indian_festival_lead_1', 'great_indian_festival_lead_2'], inplace = True)

In [502]:
final_df.columns

Index(['key', 'month_date', 'pred_SARIMA', 'pred_p3m', 'pred_p6m',
       'pred_prophet', 'pred_rf', 'pred_value_SARIMA', 'pred_value_p3m',
       'pred_value_p6m', 'pred_value_prophet', 'pred_value_rf',
       'parent_material_code', 'brand_code', 'ratio_last_year', 'quarter',
       'qtr_ind_rate_x', 'vol_in_rum_value', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', 'OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2',
       'OT_Value_in_C

In [503]:
missing_df['LY P3M'].sum()

0.0

In [504]:
final_df = final_df.sort_values(['key', 'month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key'])['vol_in_rum_value']
    .shift(10)
)


In [505]:
final_df[final_df['month_date'] == '2026-08-31']['P3M_value'].sum()

200.75072579246722

In [506]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [507]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,709538,2026-03-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,709538,2026-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,709538,2026-05-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,709538,2026-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,709538,2026-07-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
4985,811416,2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4986,811416,2026-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4987,811416,2026-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4988,811416,2026-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [508]:
final_df.shape

(188446, 82)

In [ ]:
#adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key','run_month'],
    how="left"
)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-03-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-05-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-07-

In [510]:
final_df.rename(columns = {"run_month_x":'run_month'}, inplace = True)

In [511]:
all_brand = final_df.groupby(['brand_code', 'run_month','month_date'])['vol_in_rum_value'].sum().reset_index()
all_brand

,brand_code,run_month,month_date,vol_in_rum_value
0,ADV-AHO-R,2026-03-31,2023-01-31,1.918636
1,ADV-AHO-R,2026-03-31,2023-02-28,1.495265
2,ADV-AHO-R,2026-03-31,2023-03-31,1.416496
3,ADV-AHO-R,2026-03-31,2023-04-30,1.345674
4,ADV-AHO-R,2026-03-31,2023-05-31,1.212887
...,...,...,...,...
26797,VEG_CLEAN,2026-07-31,2026-10-31,0.000000
26798,VEG_CLEAN,2026-07-31,2026-11-30,0.000000
26799,VEG_CLEAN,2026-07-31,2026-12-31,0.000000
26800,VEG_CLEAN,2026-07-31,2027-01-31,0.000000


In [512]:
all_brand.dtypes

brand_code                  object
run_month           datetime64[ns]
month_date          datetime64[ns]
vol_in_rum_value           float64
dtype: object

In [513]:
import pandas as pd


def detect_month_anomaly(
    df,
    brand_code,
    threshold=0.25,
    months_window=3,
    future_months=4
):
    """
    Logic:

    1. Get run_month dynamically.
    2. Analyze the next 4 months after run_month.
    3. For each future month:
         - Look at that month's value in Last Year (LY)
         - Compare it with the 3 months immediately preceding
           the run_month in LY.
    4. Do the exact same comparison for 2 Years Ago.
    5. A future month is finalized as an anomaly only when
       BOTH LY and 2Y-ago show the anomaly.
    6. For a given year, all_df 3 percentage differences must:
         - exceed 25%
         - have the same direction
    """

    # =========================================================
    # 1. Filter brand
    # =========================================================

    df_brand = df[
        df['brand_code'] == brand_code
    ].copy()

    if df_brand.empty:
        return None, False

    # =========================================================
    # 2. Convert dates
    # =========================================================

    df_brand['month_date'] = pd.to_datetime(
        df_brand['month_date']
    )

    df_brand['run_month'] = pd.to_datetime(
        df_brand['run_month']
    )

    # Normalize month_date to month-end so that
    # comparisons work consistently.
    df_brand['month_date'] = (
        df_brand['month_date']
        + pd.offsets.MonthEnd(0)
    )

    # =========================================================
    # 3. Determine run month dynamically
    # =========================================================

    run_month = df_brand['run_month'].max()

    run_month = (
        pd.Timestamp(run_month)
        + pd.offsets.MonthEnd(0)
    )

    # =========================================================
    # 4. Determine next 4 months
    # =========================================================

    target_dates = [
        run_month + pd.DateOffset(months=i)
        for i in range(1, future_months + 1)
    ]

    # Example:
    #
    # run_month = Sep 2026
    #
    # target_dates:
    # Oct 2026
    # Nov 2026
    # Dec 2026
    # Jan 2027

    # =========================================================
    # 5. Historical years
    # =========================================================

    # For each target month we evaluate:
    #
    # LY
    # 2Y ago
    #
    # Example:
    #
    # Oct 2026
    #   -> Oct 2025
    #   -> Oct 2024

    historical_offsets = [1, 2]

    results = []

    # =========================================================
    # 6. Process each future month
    # =========================================================

    for target_date in target_dates:

        year_results = {}

        # -----------------------------------------------------
        # For LY and 2Y ago
        # -----------------------------------------------------

        for year_offset in historical_offsets:

            # -------------------------------------------------
            # Target historical date
            # -------------------------------------------------

            historical_target_date = (
                target_date
                - pd.DateOffset(years=year_offset)
                + pd.offsets.MonthEnd(0)
            )
            print('historical_target_date',historical_target_date)

            # -------------------------------------------------
            # Fixed comparison months
            #
            # These are based on the run month.
            #
            # Example:
            #
            # run_month = Sep 2026
            #
            # LY:
            # target = Oct 2025
            # comparison = Jul, Jun, May 2025
            #
            # 2Y:
            # target = Oct 2024
            # comparison = Jul, Jun, May 2024
            # -------------------------------------------------

            comparison_dates = [
                (
                    run_month
                    - pd.DateOffset(months=i)
                    - pd.DateOffset(years=year_offset)
                    + pd.offsets.MonthEnd(0)
                )
                for i in range(1, months_window + 1)
            ]

            # -------------------------------------------------
            # Get target value
            # -------------------------------------------------

            target_value_series = df_brand.loc[
                df_brand['month_date'] == historical_target_date,
                'vol_in_rum_value'
            ]

            if target_value_series.empty:
                year_results[year_offset] = None
                continue

            target_value = target_value_series.iloc[0]

            # -------------------------------------------------
            # Get comparison values
            # -------------------------------------------------

            comparison_values = []

            missing_comparison = False

            for comparison_date in comparison_dates:

                value_series = df_brand.loc[
                    df_brand['month_date'] == comparison_date,
                    'vol_in_rum_value'
                ]

                if value_series.empty:
                    missing_comparison = True
                    break

                comparison_values.append(
                    value_series.iloc[0]
                )

            if missing_comparison:
                year_results[year_offset] = None
                continue

            # -------------------------------------------------
            # Calculate percentage differences
            # -------------------------------------------------

            pct_diffs = []

            for comparison_value in comparison_values:

                if comparison_value == 0:
                    continue

                pct_diff = (
                    (target_value - comparison_value)
                    / comparison_value
                )

                pct_diffs.append(pct_diff)

            # Need all_df 3 comparisons
            if len(pct_diffs) != months_window:

                year_results[year_offset] = None
                continue

            # -------------------------------------------------
            # Check same direction
            # -------------------------------------------------

            all_positive = all(
                p > 0 for p in pct_diffs
            )

            all_negative = all(
                p < 0 for p in pct_diffs
            )

            same_direction = (
                all_positive or all_negative
            )

            # -------------------------------------------------
            # Check threshold
            # -------------------------------------------------

            all_above_threshold = all(
                abs(p) > threshold
                for p in pct_diffs
            )

            # -------------------------------------------------
            # Final anomaly for this historical year
            # -------------------------------------------------

            is_anomaly = (
                same_direction
                and all_above_threshold
            )

            direction = None

            if all_positive:
                direction = 'higher'

            elif all_negative:
                direction = 'lower'

            # -------------------------------------------------
            # Store year result
            # -------------------------------------------------

            year_results[year_offset] = {
                'historical_target_date': historical_target_date,
                'target_value': target_value,

                'comparison_dates': comparison_dates,
                'comparison_values': comparison_values,

                'pct_diffs': pct_diffs,

                'min_pct_diff': min(pct_diffs) * 100,
                'max_pct_diff': max(pct_diffs) * 100,

                'same_direction': same_direction,
                'all_above_threshold': all_above_threshold,

                'is_anomaly': is_anomaly,
                'direction': direction
            }

        # =====================================================
        # 7. Final anomaly requires BOTH years
        # =====================================================

        ly_result = year_results.get(1)
        two_year_result = year_results.get(2)

        pattern_repeats = (
            ly_result is not None
            and two_year_result is not None
            and ly_result['is_anomaly']
            and two_year_result['is_anomaly']
        )

        # =====================================================
        # 8. Store final result
        # =====================================================

        row = {
            'brand_code': brand_code,

            'run_month': run_month,

            'target_month': target_date,

            'target_year': target_date.year,

            # -----------------------------
            # Last Year
            # -----------------------------

            'ly_target_date': (
                ly_result['historical_target_date']
                if ly_result else None
            ),

            'ly_target_value': (
                ly_result['target_value']
                if ly_result else None
            ),

            'ly_comparison_dates': (
                ly_result['comparison_dates']
                if ly_result else None
            ),

            'ly_comparison_values': (
                ly_result['comparison_values']
                if ly_result else None
            ),

            'ly_pct_diffs': (
                ly_result['pct_diffs']
                if ly_result else None
            ),

            'ly_min_pct_diff': (
                ly_result['min_pct_diff']
                if ly_result else None
            ),

            'ly_max_pct_diff': (
                ly_result['max_pct_diff']
                if ly_result else None
            ),

            'ly_direction': (
                ly_result['direction']
                if ly_result else None
            ),

            'ly_is_anomaly': (
                ly_result['is_anomaly']
                if ly_result else False
            ),

            # -----------------------------
            # 2 Years Ago
            # -----------------------------

            'two_year_target_date': (
                two_year_result['historical_target_date']
                if two_year_result else None
            ),

            'two_year_target_value': (
                two_year_result['target_value']
                if two_year_result else None
            ),

            'two_year_comparison_dates': (
                two_year_result['comparison_dates']
                if two_year_result else None
            ),

            'two_year_comparison_values': (
                two_year_result['comparison_values']
                if two_year_result else None
            ),

            'two_year_pct_diffs': (
                two_year_result['pct_diffs']
                if two_year_result else None
            ),

            'two_year_min_pct_diff': (
                two_year_result['min_pct_diff']
                if two_year_result else None
            ),

            'two_year_max_pct_diff': (
                two_year_result['max_pct_diff']
                if two_year_result else None
            ),

            'two_year_direction': (
                two_year_result['direction']
                if two_year_result else None
            ),

            'two_year_is_anomaly': (
                two_year_result['is_anomaly']
                if two_year_result else False
            ),

            # -----------------------------
            # FINAL
            # -----------------------------

            'pattern_repeats': pattern_repeats,

            'is_final_anomaly': pattern_repeats
        }

        results.append(row)

    # =========================================================
    # 9. Return
    # =========================================================

    result_df = pd.DataFrame(results)

    return result_df, result_df['is_final_anomaly'].any()


# =============================================================
# RUN FOR ALL BRANDS
# =============================================================

brands = all_brand['brand_code'].unique()

results = []

for brand in brands:

    df_result, repeats = detect_month_anomaly(
        all_brand,
        brand_code=brand,
        threshold=0.25,
        months_window=3,
        future_months=4
    )

    if df_result is not None and not df_result.empty:
        results.append(df_result)


# =============================================================
# FINAL OUTPUT
# =============================================================

if results:

    anomaly_summary = pd.concat(
        results,
        ignore_index=True
    )

else:

    anomaly_summary = pd.DataFrame()

historical_target_date 2025-08-31 00:00:00
historical_target_date 2024-08-31 00:00:00
historical_target_date 2025-09-30 00:00:00
historical_target_date 2024-09-30 00:00:00
historical_target_date 2025-10-31 00:00:00
historical_target_date 2024-10-31 00:00:00
historical_target_date 2025-11-30 00:00:00
historical_target_date 2024-11-30 00:00:00
historical_target_date 2025-08-31 00:00:00
historical_target_date 2024-08-31 00:00:00
historical_target_date 2025-09-30 00:00:00
historical_target_date 2024-09-30 00:00:00
historical_target_date 2025-10-31 00:00:00
historical_target_date 2024-10-31 00:00:00
historical_target_date 2025-11-30 00:00:00
historical_target_date 2024-11-30 00:00:00
historical_target_date 2025-08-31 00:00:00
historical_target_date 2024-08-31 00:00:00
historical_target_date 2025-09-30 00:00:00
historical_target_date 2024-09-30 00:00:00
historical_target_date 2025-10-31 00:00:00
historical_target_date 2024-10-31 00:00:00
historical_target_date 2025-11-30 00:00:00
historical_

In [514]:
all_brand

,brand_code,run_month,month_date,vol_in_rum_value
0,ADV-AHO-R,2026-03-31,2023-01-31,1.918636
1,ADV-AHO-R,2026-03-31,2023-02-28,1.495265
2,ADV-AHO-R,2026-03-31,2023-03-31,1.416496
3,ADV-AHO-R,2026-03-31,2023-04-30,1.345674
4,ADV-AHO-R,2026-03-31,2023-05-31,1.212887
...,...,...,...,...
26797,VEG_CLEAN,2026-07-31,2026-10-31,0.000000
26798,VEG_CLEAN,2026-07-31,2026-11-30,0.000000
26799,VEG_CLEAN,2026-07-31,2026-12-31,0.000000
26800,VEG_CLEAN,2026-07-31,2027-01-31,0.000000


In [515]:
anomaly_summary.rename(columns={'target_month':'month_date'}, inplace=True)
# anomaly_summary.rename(columns={'month':'month_date'}, inplace=True)

anomaly_summary

,brand_code,run_month,month_date,target_year,ly_target_date,ly_target_value,ly_comparison_dates,ly_comparison_values,ly_pct_diffs,ly_min_pct_diff,ly_max_pct_diff,ly_direction,ly_is_anomaly,two_year_target_date,two_year_target_value,two_year_comparison_dates,two_year_comparison_values,two_year_pct_diffs,two_year_min_pct_diff,two_year_max_pct_diff,two_year_direction,two_year_is_anomaly,pattern_repeats,is_final_anomaly
0,ADV-AHO-R,2026-07-31,2026-08-31,2026,2025-08-31,0.934241,"[2025-06-30 00:00:00, 2025-05-31 00:00:00, 202...","[0.9049084481942964, 1.1063053159928662, 1.012...","[0.0324151600169103, -0.15553040662674283, -0....",-15.553041,3.241516,None,False,2024-08-31,0.965291,"[2024-06-30 00:00:00, 2024-05-31 00:00:00, 202...","[1.1227043813139008, 1.288207877178269, 1.3100...","[-0.1402093380108575, -0.25067160329558563, -0...",-26.317248,-14.020934,lower,False,False,False
1,ADV-AHO-R,2026-07-31,2026-09-30,2026,2025-09-30,1.252979,"[2025-06-30 00:00:00, 2025-05-31 00:00:00, 202...","[0.9049084481942964, 1.1063053159928662, 1.012...","[0.38464727101028345, 0.13257976364497787, 0.2...",13.257976,38.464727,higher,False,2024-09-30,1.068057,"[2024-06-30 00:00:00, 2024-05-31 00:00:00, 202...","[1.1227043813139008, 1.288207877178269, 1.3100...","[-0.04867479369883953, -0.17089702982707342, -...",-18.472876,-4.867479,lower,False,False,False
2,ADV-AHO-R,2026-07-31,2026-10-31,2026,2025-10-31,1.031031,"[2025-06-30 00:00:00, 2025-05-31 00:00:00, 202...","[0.9049084481942964, 1.1063053159928662, 1.012...","[0.13937585443924058, -0.06804132507909585, 0....",-6.804133,13.937585,None,False,2024-10-31,1.227544,"[2024-06-30 00:00:00, 2024-05-31 00:00:00, 202...","[1.1227043813139008, 1.288207877178269, 1.3100...","[0.09338121489505996, -0.04709177598117922, -0...",-6.298892,9.338121,None,False,False,False
3,ADV-AHO-R,2026-07-31,2026-11-30,2026,2025-11-30,0.918745,"[2025-06-30 00:00:00, 2025-05-31 00:00:00, 202...","[0.9049084481942964, 1.1063053159928662, 1.012...","[0.015290687778979512, -0.16953746179945087, -...",-16.953746,1.529069,None,False,2024-11-30,0.896491,"[2024-06-30 00:00:00, 2024-05-31 00:00:00, 202...","[1.1227043813139008, 1.288207877178269, 1.3100...","[-0.20148955024742848, -0.3040787932256338, -0...",-31.568869,-20.148955,lower,False,False,False
4,BIO OILS,2026-07-31,2026-08-31,2026,2025-08-31,2.626375,"[2025-06-30 00:00:00, 2025-05-31 00:00:00, 202...","[2.8603744870328955, 3.2579445935414832, 2.812...","[-0.08180715866474068, -0.1938551125951867, -0...",-19.385511,-6.624688,lower,False,2024-08-31,3.192634,"[2024-06-30 00:00:00, 2024-05-31 00:00:00, 202...","[3.168412192190167, 3.718361745537151, 3.33935...","[0.007644683714670177, -0.14138700864459308, -...",-14.138701,0.764468,None,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527,SW_SGPRF,2026-07-31,2026-11-30,2026,2025-11-30,0.585369,"[2025-06-30 00:00:00, 2025-05-31 00:00:00, 202...","[1.4217706982993206, 1.7278646501013846, 1.291...","[-0.5882820593124383, -0.6612185427837963, -0....",-66.121854,-54.666631,lower,True,2024-11-30,0.463842,"[2024-06-30 00:00:00, 2024-05-31 00:00:00, 202...","[2.066429599084488, 1.4518611740625516, 0.8112...","[-0.7755345599204375, -0.6805190209377765, -0....",-77.553456,-42.824573,lower,True,True,True
528,VEG_CLEAN,2026-07-31,2026-08-31,2026,NaT,NaN,None,None,None,NaN,NaN,None,False,NaT,NaN,None,None,None,NaN,NaN,None,False,False,False
529,VEG_CLEAN,2026-07-31,2026-09-30,2026,NaT,NaN,None,None,None,NaN,NaN,None,False,NaT,NaN,None,None,None,NaN,NaN,None,False,False,False
530,VEG_CLEAN,2026-07-31,2026-10-31,2026,NaT,NaN,None,None,None,NaN,NaN,None,False,NaT,NaN,None,None,None,NaN,NaN,None,False,False,False


In [516]:

# def detect_month_anomaly(df, brand_code, month_num, mon=9,threshold=0.25, months_window=3):
#     """
#     Detect if a specific month's vol_in_rum_value is >25% different 
#     from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
#     Parameters:
#     - df: input dataframe with 'month_date', 'vol_in_rum_value'
#     - brand_code: filter by this brand code
#     - month_num: month to check (6, 7, 8, 9)
#     - threshold: 25% difference threshold
#     - months_window: number of months before and after to compare
#     """
    
#     df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
#     if df_brand.empty:
#         return None
    
#     df_brand['year'] = df_brand['month_date'].dt.year
#     df_brand['month'] = df_brand['month_date'].dt.month
    
#     years = sorted(df_brand['year'].unique())
#     current_year = years[-1]
#     past_years = [y for y in years if y < current_year][-2:]
    
#     anomalies = []
    
#     for year in past_years:
#         df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
#         month_data = df_year[df_year['month'] == month_num]
#         if month_data.empty:
#             continue
        
#         month_value = month_data['vol_in_rum_value'].iloc[0]
        
#         past_months = [(mon - i - 1) % 12 for i in range(1, months_window + 1)]
#         print(past_months)
#         next_months = [(mon + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        
#         past_m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
#         next_m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
#         comparison_values = pd.concat([past_m])
        
#         if comparison_values.empty:
#             continue
        
#         pct_diffs = []
#         for comp_value in comparison_values:
#             if comp_value != 0:
#                 pct_diff = (month_value - comp_value) / comp_value
#                 pct_diffs.append(pct_diff)
        
#         if pct_diffs:
#             positive_diffs = [p for p in pct_diffs if p > 0]
#             negative_diffs = [p for p in pct_diffs if p < 0]
#             same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
#             is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
#         else:
#             is_anomaly = False

#         anomalies.append({
#             'brand_code': brand_code,
#             'month': month_num,
#             'year': year,
#             'month_value': month_value,
#             'num_months_compared': len(comparison_values),
#             'pct_diffs_from_each': pct_diffs,
#             'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
#             'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
#             'is_anomaly': is_anomaly,
#             'direction': 'higher' if month_value > comparison_values.mean() else 'lower'
#         })
    
#     if len(anomalies) == 2:
#         pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
#         return pd.DataFrame(anomalies), pattern_repeats
    
#     return pd.DataFrame(anomalies), False


# # Check months 6, 7, 8, 9
# brands = all_brand['brand_code'].unique()
# results = []

# for month in [9,10,11,12]:
#     for brand in brands:
#         df_result, repeats = detect_month_anomaly(all_brand, brand, month)
#         if df_result is not None and not df_result.empty:
#             df_result['pattern_repeats'] = repeats
#             results.append(df_result)

# anomaly_summary = pd.concat(results, ignore_index=True)
# # print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

In [517]:
final_df#.shape

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-03-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-05-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-07-31

In [518]:
xx = final_df.copy()

In [519]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code','month_date'])[['brand_code', 'month_date','two_year_direction']]
final_brands['month_different'] = 1
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(final_brands[['brand_code', 'month_date','month_different']], on = ['brand_code','month_date'], how = 'left')
final_df['month_different'].fillna(0, inplace=True)
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-03-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-05-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN

In [520]:
# final_df[final_df['month_different'] == 1]

In [521]:
# final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code'])[['brand_code', 'direction', 'min_pct_diff', 'max_pct_diff']]
# final_brands['month_different'] = 1
# final_df = final_df.merge(final_brands[['brand_code', 'month_different']], on = 'brand_code', how = 'left')
# final_df['month_different'].fillna(0, inplace=True)
# final_df

In [526]:
final_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,brand_code,ratio_last_year,quarter,qtr_ind_rate_x,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,event_month_flag,event_uplift_factor,event_sensitive_flag,P3M_adj,P6M_adj,P3M_adj_value,P6M_adj_value,LY_P3M_adj,LY_P6M_adj,LY_P3M_adj_value,LY_P6M_adj_value,qtr_ind_rate_y,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-03-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-05-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN,NaN,10,0.0,NaN,0.0,0,2026-06-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,709538,2023-10-31,0.0,0.06,0.04,0.089103,0.054678,0.0,0.00001,0.000007,0.000015,0.000009,709538,LIVON SLP,0.0,4.0,1634.657242,0.000016,0.1,0.000016,2026-02-28,2.703246,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-03-31,None,PREM. HAIR NOUR.,0.094045,0.09136,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,0.000015,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1634.657242,0,0,0.0,0.0,NaN,NaN,NaN

In [535]:
offtake_df[offtake_df['month_date']>'2026-03-31'].groupby(['month_date'])['indexbpm'].sum()

month_date
2026-04-30    3553.65100
2026-05-31    4196.73659
2026-06-30    3767.55961
2026-07-31    4251.68642
2026-08-31    4378.88980
2026-09-30       0.00000
2026-10-31       0.00000
2026-11-30       0.00000
2026-12-31       0.00000
2027-01-31       0.00000
2027-02-28       0.00000
2027-03-31       0.00000
2027-04-30       0.00000
2027-05-31       0.00000
2027-06-30       0.00000
Name: indexbpm, dtype: float64

In [530]:
final_df[final_df['month_date']>'2026-03-31'].groupby(['run_month','month_date'])[['vol_in_rum_value','pred_value_prophet','P3M_value']].sum()

vol_in_rum_value  pred_value_prophet   P3M_value
run_month  month_date                                                  
2026-03-31 2026-04-30        177.654917          164.746414  207.414347
           2026-05-31        209.803960          172.531667  207.414347
           2026-06-30        188.340884          179.411163  207.414347
           2026-07-31        212.531809          224.765480  207.414347
           2026-08-31        218.880078          213.986290  207.414347
           2026-09-30          0.000000          228.008231  207.414347
           2026-10-31          0.000000          330.429304  207.414347
           2026-11-30          0.000000            0.000000   22.929993
           2026-12-31          0.000000            0.000000   22.929993
2026-04-30 2026-04-30        177.654917          162.844084  207.361950
           2026-05-31        209.803960          171.762488  207.361950
           2026-06-30        188.340884          176.424745  207.361950
           2026-07-31        212.531809          226.766932  207.361950
           2026-08-31        218.880078          221.005269  207.361950
           2026-09-30          0.000000          217.236874  207.361950
           2026-10-31          0.000000          329.233503  207.361950
           2026-11-30          0.000000          217.524216  207.361950
           2026-12-31          0.000000            0.000000   27.276597
           2027-01-31          0.000000            0.000000   27.276597
2026-05-31 2026-04-30        177.654917          158.882100  207.361950
           2026-05-31        209.803960          172.338503  194.310794
           2026-06-30        188.340884          179.465009  194.310794
           2026-07-31        212.531809          221.119801  194.310794
           2026-08-31        218.880078          218.411466  194.310794
           2026-09-30          0.000000          220.435127  194.310794
           2026-10-31          0.000000          337.184905  194.310794
           2026-11-30          0.000000          214.528056  194.310794
           2026-12-31          0.000000          217.098486  194.310794
           2027-01-31          0.000000            0.000000   27.678186
           2027-02-28          0.000000            0.000000   27.678186
2026-06-30 2026-04-30        177.654917          161.167410  207.361950
           2026-05-31        209.803960          173.497506  194.310794
           2026-06-30        188.340884          179.060322  202.539328
           2026-07-31        212.531809          254.580863  202.539328
           2026-08-31        218.880078          217.594674  202.539328
           2026-09-30          0.000000          243.247636  202.539328
           2026-10-31          0.000000          318.653604  202.539328
           2026-11-30          0.000000          219.819569  202.539328
           2026-12-31          0.000000          217.263281  202.539328
           2027-01-31          0.000000          195.309702  202.539328
           2027-02-28          0.000000            0.000000   28.350340
           2027-03-31          0.000000            0.000000   28.350340
2026-07-31 2026-04-30        177.654917          160.302186  207.361950
           2026-05-31        209.803960          171.650217  194.310794
           2026-06-30        188.340884          170.880308  202.539328
           2026-07-31        212.531809          258.629290  192.127209
           2026-08-31        218.880078          209.184238  192.127209
           2026-09-30          0.000000          242.266069  192.127209
           2026-10-31          0.000000          319.052111  192.127209
           2026-11-30          0.000000          215.063507  192.127209
           2026-12-31          0.000000          215.082212  192.127209
           2027-01-31          0.000000          191.841145  192.127209
           2027-02-28          0.000000          171.480435  192.127209
           2027-03-31          0.000000            0.000000   2

In [532]:
final_df[(final_df['M month'].notna())].to_csv(f'/data/aman_singh/acuuracy_check/all_combination_ecom_bactest_pred.csv')

In [166]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_ecom_trend.csv')

In [154]:
final_df[final_df['month_date'] == '2026-10-31']['pred_value_prophet'].sum()

50.02404523409474

### some checks

In [155]:
query = f"""select * from {input_table}
where run_month = '{month_run}' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month
0,2026-08-31,Amazon ARIPL,718288,40.992,56.923648,SAFF GOLD,0,0,0,0,1,0,0,0,0,1,0,NaN,3,2026-09-30
1,2026-09-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,1,0,0,0,0,1,0,0,0,0,NaN,3,2026-09-30
2,2026-10-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,1,0,0,0,0,1,0,0,0,0.0,4,2026-09-30
3,2026-11-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,1,0,0,0,0,1,0,0,0.0,4,2026-09-30
4,2026-12-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,0,0,0,0,0,0,0,0,0.0,4,2026-09-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141752,2027-02-28,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,NaN,1,2026-09-30
141753,2027-03-31,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,NaN,1,2026-09-30
141754,2027-04-30,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,0.0,2,2026-09-30
141755,2027-05-31,Purplle,810674,0.000,0.000000,PA_ESS_HO,1,0,0,0,0,0,0,0,0,0,0,0.0,2,2026-09-30


In [156]:
data['key'] = (
    data['platform_name'].astype(str) + '_' +
    data['parent_material_code'].astype(str)
)
data = data[data['key'].isin(trend_file_df['key'].unique())]

In [157]:
trend_file_df

,key,month_date,pred_SARIMA,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_SARIMA,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,platform_name,brand_code,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,ratio_last_year,quarter,qtr_ind_rate,vol_in_rum_value,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,Amazon RK_718472,2023-11-30,0.000000e+00,0.51,1.125,NaN,0.4590,0.000000e+00,0.000025,0.000056,NaN,0.000023,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,0.000000,4,496.828458,0.000022,0.45,0.000022,2026-08-31,2.702665,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,1.0,0.0,0.0,0.0,2026-09-30,None,HAIR OILS,NaN,NaN,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon RK_718472,2023-12-31,4.499998e-01,0.51,1.125,NaN,0.3690,2.235727e-05,0.000025,0.000056,NaN,0.000018,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,NaN,4,496.828458,0.000000,0.00,0.000000,2026-08-31,2.702665,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,1.0,0.0,0.0,2026-09-30,None,HAIR OILS,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000022,NaN,NaN
2,Amazon RK_718472,2024-01-31,1.635320e-01,0.51,1.125,NaN,0.9486,8.124736e-06,0.000025,0.000056,NaN,0.000047,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,2.400000,1,496.828458,0.000054,1.08,0.000054,2026-08-31,2.702665,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-09-30,None,HAIR OILS,NaN,NaN,1.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000022,NaN
3,Amazon RK_718472,2024-02-29,7.076602e-01,0.51,1.125,NaN,1.4706,3.515857e-05,0.000025,0.000056,NaN,0.000073,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,7.200000,1,496.828458,0.000080,1.62,0.000080,2026-08-31,2.702665,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-09-30,None,HAIR OILS,NaN,NaN,1.62,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.51,0.000025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000054,0.000000,0.000022
4,Amazon RK_718472,2024-03-31,8.456983e-01,0.90,1.125,NaN,2.0844,4.201670e-05,0.000045,0.000056,NaN,0.000104,718472,Amazon RK,ADV-AHO-R,NaN,NaN,NaN,NaN,NaN,5.294118,1,496.828458,0.000134,2.70,0.000134,2026-08-31,2.702665,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,0.0,0.0,0.0,0.0,0.0,2026-09-30,None,HAIR OILS,NaN,NaN,2.70,0.90,NaN,NaN,NaN,NaN,NaN,NaN,76.470588,NaN,NaN,NaN,0.90,0.000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000080,0.000054,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126020,Big Basket_719192,2026-12-31,-8.050627e-311,0.00,0.000,NaN,0.0000,-8.050627e-316,0.000000,0.000000,NaN,0.000000,719192,Big Basket,VEG_CLEAN,0.0,0.0,0.0,0.0,0.0,NaN,4,100.000000,0.000000,0.00,0.000000,2026-08-31,4.783100,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,NaN,NaN,NaN,NaN,NaN,2026-09-30,M+3,HEALTH & HYGIENE,NaN,NaN,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.000000,-100.0,-100.0,0.0,0.00

In [158]:
import pandas as pd

as_of_date = pd.to_datetime(month_run)  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [159]:
# assert p3m equals
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == month_run]['P3M'].sum()
assert(int(x)==int(y))

In [160]:
(x,y)

(266474.77709504537, 266474.7770950415)

In [161]:
ly_end = as_of_date - pd.DateOffset(years=1)
ly_start = ly_end - pd.DateOffset(months=3)

filtered = data[
    
    (data['month_date'] < ly_end) &
    (data['month_date'] >= ly_start )
]


In [162]:
# p3m ly check may not equal but should be close
x = filtered.groupby(['month_date'])['vol_in_rum'].sum().reset_index()['vol_in_rum'].mean()
y = trend_file_df[trend_file_df['month_date'] == month_run]['LY P3M'].sum()
(x,y)

(270739.84984885, 270374.1616822037)

In [163]:
# p3m consistency check
as_of_date = pd.to_datetime(month_run)

next_3_months = pd.date_range(
    start=as_of_date + pd.offsets.MonthEnd(1),
    periods=3,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = trend_file_df.loc[
        trend_file_df['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-10-31: 266474.7770950415
P3M sum for 2026-11-30: 266474.7770950415
P3M sum for 2026-12-31: 266474.7770950415


### The end

In [106]:
import numpy as np
import pandas as pd

df = final_df.copy()

# --------------------------------------------------
# 1. SAFE FACTOR FUNCTIONS (NO ERRORS)
# --------------------------------------------------

def safe_div(a, b):
    """Safe division: if error or b<=0 → return 1."""
    try:
        if b is None or b == 0:
            return 1
        return a / b
    except:
        return 1

# recency factor = p3m/p6m (cap at 2)
df["recency_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["P6M_value"])),
    axis=1
)

def safe_shrink(r,column_name):
    try:
        ratio = r[column_name] / r["P3M_value"]
        return 1 / np.sqrt(ratio)
    except:
        return 1

df["shrink_ratio_prophet"] = df.apply(lambda r: safe_shrink(r, "pred_value_prophet"), axis=1)
df["shrink_ratio_rf"] = df.apply(lambda r: safe_shrink(r, "pred_value_rf"), axis=1)

# seasonality factor = p3m / p3mLY (cap at 2)
df["seasonality_factor"] = df.apply(
    lambda r: min(2, safe_div(r["P3M_value"], r["LY P3M_value"])),
    axis=1
)


# shrink_ratio = 1 / sqrt(max(forecast/p3m,1)) → safe



# --------------------------------------------------
# 2. HEURISTICS
# --------------------------------------------------

# Recency heuristic: max(recency_factor * P3M, forecast)
df["recency_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet_value"] = df["pred_value_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet_value"]


df["final_heuristic_prophet_value"] = df.apply(apply_final_logic, axis=1)


In [107]:
df["recency_heuristic_rf_value"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M_value"], r["pred_value_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'], r["pred_value_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf_value"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY value'] * r["recency_factor"],
                  r["pred_value_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf_value"] = df["pred_value_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf_value"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf_value"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf_value"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf_value"]


df["final_heuristic_rf_value"] = df.apply(apply_final_logic, axis=1)


In [108]:
df["error_prophet"] = df["pred_value_prophet"] - df["vol_in_rum_value"]
df["abs_error_prophet"] = df["error_prophet"].abs()

df["error_rf"] = df["pred_value_rf"] - df["vol_in_rum_value"]
df["abs_error_rf"] = df["error_rf"].abs()

df["error_final_heuristic_prophet"] = df["final_heuristic_prophet_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_prophet"] = df["error_final_heuristic_prophet"].abs()

df["error_final_heuristic_rf"] = df["final_heuristic_rf_value"] - df["vol_in_rum_value"]
df["abs_error_final_heuristic_rf"] = df["error_final_heuristic_rf"].abs()


In [109]:
df["recency_heuristic_rf"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_rf"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_rf"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_rf"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_rf"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_rf"] = df["pred_rf"] * df["shrink_ratio_rf"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_rf"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_rf"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_rf"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_rf"]


df["final_heuristic_rf"] = df.apply(apply_final_logic, axis=1)


In [110]:
df["recency_heuristic_prophet"] = df.apply(
    lambda r: max(r["recency_factor"] * r["P3M"], r["pred_prophet"]),
    axis=1
)

# Seasonality heuristic: max(seasonality_value, forecast)
df["seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'], r["pred_prophet"]),
    axis=1
)

# Recency + Seasonality combined
df["recency_seasonality_heuristic_prophet"] = df.apply(
    lambda r: max(r["seasonality_factor"]*r['LY'] * r["recency_factor"],
                  r["pred_prophet"]),
    axis=1
)

# Non-seasonal heuristic
df["non_seasonal_heuristic_prophet"] = df["pred_prophet"] * df["shrink_ratio_prophet"]


# --------------------------------------------------
# 3. FINAL DECISION TREE + skipped condition
# --------------------------------------------------

def apply_final_logic(r):

    # Rule 1: skipped → force P3M
    if r.get("skipped", 0) == 1:
        return r["P3M_value"]

    
    # Rule 3: Heuristic combinations
    if r["final_trend"] == 1 and r["seasonality_flag"] == 0:
        return r["recency_heuristic_prophet"]


    if r["final_trend"] != 1 and r["seasonality_flag"] == 1:
        return r["seasonality_heuristic_prophet"]
    
    if r["final_trend"] == 1 and r["seasonality_flag"] == 1:
        return r["recency_seasonality_heuristic_prophet"]

    # Default: (0,0)
    return r["non_seasonal_heuristic_prophet"]


df["final_heuristic_prophet"] = df.apply(apply_final_logic, axis=1)


In [112]:
df.to_csv('Heuristics_all_combination_ecom_cp.csv')

In [114]:
df[(df['M month'].notna())].to_csv('Heuristics_all_combination_ecom_cp2.csv')

In [2]:
import pandas as pd
final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')

/tmp/ipykernel_2052077/3783379672.py:2: DtypeWarning: Columns (20,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv('Heuristics_all_combination_ecom_cp.csv')


In [6]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,Amazon ARIPL,718288,2025-12-31,0.134916,0.345766,0.219256,0.042170
1,Amazon ARIPL,718321,2025-12-31,0.000000,0.000000,0.000000,0.000000
2,Amazon ARIPL,718322,2025-12-31,0.060035,0.108314,0.079347,0.009656
3,Amazon ARIPL,718323,2025-12-31,0.000000,0.000000,0.000000,0.000000
4,Amazon ARIPL,718328,2025-12-31,0.016588,0.032115,0.022799,0.003105


In [7]:
threshold_df.to_csv('ecom_threshold.csv')

In [162]:
final_df[(final_df['M month'].notna())].to_csv('Heuristics_all_combination_qcom_cp_chk.csv')

In [129]:
final_df[final_df['month_date'] == '2026-02-28']['LY P3M_value'].sum()

123.92211585241307

In [49]:
df['TREND'].unique()

array([ 1,  0, -1])

In [96]:
prophet_output = collate_file('prophet_data_train_till')

downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202508-09_FK_Others_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\new_pipeline\202509_AZ_BB_Offtakes\train_till_31_Jul_2025\prophet_results\prophet_data_train_till_31_Jul_2025.csv


In [100]:
len_before_merge = len(offtake_df)
offtake_df = offtake_df.merge(
    read_qtr_ind_rate_table()[['brand_code', 'qtr_ind_rate']] ,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(offtake_df)


Credentials retrieved successfully for prod db.


In [ ]:
offtake_df

In [104]:
offtake_df['OT_Value_in_Cr'] = offtake_df['vol_in_rum'] * offtake_df['qtr_ind_rate'] / (10 ** 7)

In [105]:
offtake_df.to_csv('Offtake_realigned_base.csv', index=False)

In [108]:
trend_file_df[trend_file_df['run_month'].isin(['2025-10-31'])].to_csv('Trend_File_Oct_Live_Run_OT.csv', index=False)

In [107]:
trend_file_df[trend_file_df['run_month'].isin(['2025-07-31', '2025-08-31'])].to_csv('Trend_File_SepAug_OT.csv', index=False)

In [97]:
prophet_output.to_csv('ECOM_Prophet_Trend_OT_Chain_PSKU_AugSep.csv', index=False)

In [81]:
forecast_df = trend_file_df[trend_file_df['month_date'] >= trend_file_df['run_month']]

forecast_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,LY P6M,P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,LY,LLY,LY value,LLY value
26,Amazon_718589,2025-03-31,1312.900000,1293.400000,1824.634404,945.661643,0.059090,0.058213,0.082122,0.042562,...,841.200000,0.059090,0.058213,0.027878,0.037860,0.082122,717.3,1428.0,0.032284,0.064271
27,Amazon_718589,2025-04-30,1312.900000,1293.400000,1894.995393,1306.938750,0.059090,0.058213,0.085289,0.058822,...,792.800000,0.059090,0.058213,0.036843,0.035682,0.085289,1053.0,1040.4,0.047393,0.046826
28,Amazon_718589,2025-05-31,1312.900000,1293.400000,2055.487818,1194.591429,0.059090,0.058213,0.092512,0.053765,...,682.250000,0.059090,0.058213,0.039944,0.030706,0.092512,1241.1,1236.3,0.055859,0.055643
29,Amazon_718589,2025-06-30,1312.900000,1293.400000,1713.411024,989.591786,0.059090,0.058213,0.077116,0.044539,...,811.600000,0.059090,0.058213,0.045178,0.036528,0.077116,797.4,1176.0,0.035889,0.052929
56,Amazon_722188,2025-03-31,0.666667,6.400000,0.000000,471.682496,0.000030,0.000288,0.000000,0.021229,...,782.533333,0.000030,0.000288,0.043045,0.035220,0.000000,806.4,236.8,0.036294,0.010658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120425,Flipkart National_809422,2025-10-31,211.833333,159.550000,262.584032,129.597491,0.030576,0.023029,0.037901,0.018706,...,207.383333,0.030576,0.023029,0.009040,0.029934,0.046132,0.8,NaN,0.000115,NaN
120442,Flipkart National_809423,2025-07-31,75.000000,53.533333,0.000000,169.044957,0.010826,0.007727,0.000000,0.024400,...,NaN,0.010826,0.007727,0.057582,NaN,0.008194,243.4,NaN,0.035132,NaN
120443,Flipkart National_809423,2025-08-31,75.000000,53.533333,0.000000,117.175693,0.010826,0.007727,0.000000,0.016913,...,NaN,0.010826,0.007727,0.054599,NaN,0.000000,99.6,NaN,0.014376,NaN
120444,Flipkart National_809423,2025-09-30,75.000000,53.533333,0.000000,111.920263,0.010826,0.007727,0.000000,0.016155,...,286.050000,0.010826,0.007727,0.048200,0.041288,0.000000,75.4,NaN,0.010883,NaN


In [98]:
trend_file_df[trend_file_df['run_month'] == '2025-08-31'].to_csv('Trend_file_Sep.csv', index=False)

In [82]:
forecast_df['is_na'] = forecast_df['vol_in_rum'].isna()
forecast_df.groupby(['run_month', 'month_date'])['is_na'].sum()

run_month   month_date
2025-03-31  2025-03-31    0
            2025-04-30    0
            2025-05-31    0
            2025-06-30    0
2025-04-30  2025-04-30    0
            2025-05-31    0
            2025-06-30    0
            2025-07-31    0
2025-05-31  2025-05-31    0
            2025-06-30    0
            2025-07-31    0
            2025-08-31    0
2025-06-30  2025-06-30    0
            2025-07-31    0
            2025-08-31    0
            2025-09-30    0
2025-07-31  2025-07-31    0
            2025-08-31    0
            2025-09-30    0
            2025-10-31    0
Name: is_na, dtype: int64

In [83]:
forecast_df.drop('is_na', axis=1, inplace=True)

In [84]:
pred_value_cols = [col for col in trend_file_df.columns if 'value' in col and 'pred' in col]
pred_value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'pred_value_best_model',
 'pred_prophet_70%ile_value']

In [85]:
for col in pred_value_cols:
    trend_file_df[f'error_{col}'] = trend_file_df[col] - trend_file_df['vol_in_rum_value']
    trend_file_df[f'abs_error_{col}'] = np.abs(trend_file_df[col] - trend_file_df['vol_in_rum_value'])    

In [86]:
trend_file_df.to_csv('Trend_file_OT_FK_AZ_BB.csv', index=False)

In [85]:
feature_importance_df = collate_file('feature_importance_train_till')

downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_28_Feb_2025\ml_results\feature_importance_train_till_28_Feb_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Apr_2025\ml_results\feature_importance_train_till_30_Apr_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_30_Jun_2025\ml_results\feature_importance_train_till_30_Jun_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_Mar_2025\ml_results\feature_importance_train_till_31_Mar_2025.csv
downloaded_results\202504-08_AZ_BB_ChainPSKU_OT_run\train_till_31_May_2025\ml_results\feature_importance_train_till_31_May_2025.csv


In [87]:
feature_importance_df.to_excel('202505-AZ_BB_OT_Feature_Imp.xlsx', index=False)

In [11]:
collate_file('prophet_data_train_till_').to_excel('202504-08_Prophet_File.xlsx', index=False)

downloaded_results\ECOM_ChainPSKU_OT_run\train_till_28_Feb_2025\prophet_results\prophet_data_train_till_28_Feb_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Apr_2025\prophet_results\prophet_data_train_till_30_Apr_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_30_Jun_2025\prophet_results\prophet_data_train_till_30_Jun_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_Mar_2025\prophet_results\prophet_data_train_till_31_Mar_2025.csv
downloaded_results\ECOM_ChainPSKU_OT_run\train_till_31_May_2025\prophet_results\prophet_data_train_till_31_May_2025.csv


### Secondary

In [87]:
sec_query = """SELECT 
    chain,
    parent_material_code, 
    month_date, 
    SUM(sec_actuals_vol_rum_month) AS sec_vol_actuals_rum_month,
    SUM(sec_apo_plan_vol_rum_month) AS sec_apo_plan_vol_rum_month
FROM (
    SELECT 
        month_date, 
        distributor_code, 
        material_code, 
        sec_actuals_vol_rum_month, 
        sec_apo_plan_vol_rum_month
    FROM 
        dwh_bpm_dist_brand_mth_sbp 
    WHERE 
        month_date BETWEEN '2022-04-01' AND '2025-12-31'
) A
JOIN (
    SELECT DISTINCT
        customer, 
        chain
    FROM 
        mst_chain_master 
    WHERE 
        chain_type = 'E Com B2C'
) CC
    ON A.distributor_Code = CC.customer
JOIN (
    SELECT 
        material_code, 
        parent_material_code 
    FROM 
        mst_material 
    WHERE 
        company_code = 'MIL' 
        AND latest_record_ind = 1
) M 
    ON A.material_code = M.material_code
GROUP BY 
    chain,
    parent_material_code, 
    month_date
ORDER BY 
    chain,
    parent_material_code, 
    month_date;
"""


results = pd.read_sql(con=prod_conn, sql=sec_query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [88]:
sales_data['month_date'] = pd.to_datetime(sales_data['month_date'])

In [89]:
sales_data['chain'] = sales_data['chain'].replace({
    'Grofers': 'Blinkit',
    'Amazon B2C': 'Amazon ARIPL',
    'Flipkart-Grocery': 'Flipkart Grocery',
    'FlipkartGrocery': 'Flipkart Grocery',
    'Flipkart-National': 'Flipkart National',
    'RK WORLDINFOCOM': 'Amazon RK',
    'ZEPTO': 'Zepto',
    'Kiranakart Technologies': 'Zepto',
    'Big basket B2B': 'Big Basket',
    'Big basket B2C': 'Big Basket',
    'Myntra': 'MYNTRA'
})

In [90]:
chains = ['Big Basket', 'Blinkit', 'Amazon ARIPL', 'Flipkart Grocery',
       'Flipkart National', 'Swiggy', 'Zepto', 'Amazon RK', 'City Mall',
       '1MG', 'Dealshare', 'Meesho', 'Nykaa', 'First Cry', 'MYNTRA',
       'Purplle']

In [91]:
for chain in chains:
    if chain not in sales_data['chain'].unique():
        print(chain)

In [92]:
dev_conn = get_dbconnection('DEV')
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment_2""",
    dev_conn
)

realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel='ECOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "All")
    ]
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']
    
    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data['parent_material_code'] == old_psku, "parent_material_code"
        ] = new_psku

    return data


Credentials retrieved successfully for dev db.


In [93]:
realigned_df = realign_pskus(sales_data.copy())

In [94]:
realigned_df = realigned_df.groupby(
    ['chain', 'parent_material_code', 'month_date'], as_index=False
).sum()

In [95]:
old_pskus = realignment_df[
    (realignment_df["channel"] == "ECOM")
    | (realignment_df["channel"] == "ECOM" + " B2C")
    | (realignment_df["channel"] == "All")
]['psku old'].unique()

for psku in old_pskus:
    assert psku not in realigned_df['parent_material_code'].unique()

In [96]:
realigned_df['key'] = realigned_df['chain'] + '_' + realigned_df['parent_material_code'].astype(str) 
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['key', 'month_date']).sum()

0

In [97]:
realigned_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [99]:
realigned_df


,chain,parent_material_code,month_date,sec_vol_actuals_rum_month,sec_apo_plan_vol_rum_month,key
0,1MG,715095,2024-01-31,0.0,0.000,1MG_715095
1,1MG,715096,2023-05-31,0.0,0.000,1MG_715096
2,1MG,715096,2023-06-30,0.0,0.189,1MG_715096
3,1MG,715096,2023-07-31,0.0,0.016,1MG_715096
4,1MG,715096,2023-08-31,0.0,0.000,1MG_715096
...,...,...,...,...,...,...
1081559,imli,807033,2025-04-30,0.0,0.000,imli_807033
1081560,imli,807033,2025-05-31,0.0,0.000,imli_807033
1081561,imli,807033,2025-06-30,0.0,0.000,imli_807033
1081562,imli,807033,2025-07-31,0.0,0.000,imli_807033


In [105]:
trend_file_df['platform_name'].unique()

array(['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'],
      dtype=object)

In [106]:
trend_file_df['platform_updated'] = np.where(
    trend_file_df['platform_name'] == 'Amazon', 
    np.where(
        trend_file_df['portfolio'].isin(['Foods', 'Saffola Oils']), 
        'Amazon ARIPL', 
        'Amazon RK'
    ),
    trend_file_df['platform_name']
)

In [108]:
trend_file_df['platform_updated'].unique()

array(['Amazon RK', 'Big Basket', 'Flipkart Grocery', 'Flipkart National',
       'Amazon ARIPL'], dtype=object)

In [109]:
trend_file_df.groupby(
    'platform_name'
)['platform_updated'].unique()

platform_name
Amazon               [Amazon RK, Amazon ARIPL]
Big Basket                        [Big Basket]
Flipkart Grocery            [Flipkart Grocery]
Flipkart National          [Flipkart National]
Name: platform_updated, dtype: object

In [102]:
trend_file_df['portfolio']

0             Hair Oils
1             Hair Oils
2             Hair Oils
3             Hair Oils
4             Hair Oils
              ...      
120441    Male Grooming
120442    Male Grooming
120443    Male Grooming
120444    Male Grooming
120445    Male Grooming
Name: portfolio, Length: 120446, dtype: object

In [113]:
trend_file_df['key'] = trend_file_df[
    ['platform_updated', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [116]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    realigned_df[['key', 'month_date', 'sec_vol_actuals_rum_month']],
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge